# GNR 640 — Mini Project: Geospatial Statistics
## USCRN Gridded Climate Data Generation & Evaluation

**Project Overview:** Generate 2° daily gridded products for 5 hydroclimatic variables using USCRN station observations, evaluate against ERA5 reanalysis, and recommend best interpolation models.

**Variables (5):** Air Temperature, Precipitation, Relative Humidity, Soil Temperature, Soil Moisture (10cm)

**Spatial Domain:** Entire United States — CONUS, Alaska, Hawaii (2006–2021)

**Methods:**
- Ordinary Kriging with 5 variogram models (spherical, exponential, gaussian, linear, circular)
- AIC + Leave-One-Out Cross-Validation for internal model selection
- Hawaii uses IDW (0.25°) due to sparse stations (n=2) — see Longman et al. (2019)
- ERA5 monthly means for external validation (RMSE, correlation, KS test)

**Deliverables (per instructions):**
1. ✅ Gridded data products (2° spatial, daily temporal) → NetCDF files
2. ✅ Performance report (RMSE + Correlation vs ERA5)
3. ✅ Statistical analysis (Central tendency, Dispersion, KS test)
4. ✅ Seasonality analysis
5. ✅ Model suitability recommendation

**Author:** 25D1388 


---
# 1. Setup & Imports

Install required packages and import all libraries used in this analysis.

In [2]:
# Install required packages
!pip install -q openpyxl pykrige xarray netcdf4 cartopy geopandas \
                shapely matplotlib tqdm scipy cdsapi

In [7]:
# Imports
import os
import time
import warnings
from datetime import datetime
from collections import Counter
from io import BytesIO

import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats as scipy_stats
from scipy.spatial.distance import pdist, squareform
from scipy.optimize import curve_fit
from scipy.interpolate import griddata

import geopandas as gpd
from shapely.geometry import Point
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.io.shapereader as shpreader

from pykrige.ok import OrdinaryKriging
from tqdm import tqdm

warnings.filterwarnings('ignore')
np.random.seed(42)

print("✅ All libraries imported")

✅ All libraries imported


In [ ]:
# Define paths
BASE_DIR = "/path"
GT_DIR = os.path.join(BASE_DIR, "groundtruth")
ERA5_DIR = os.path.join(BASE_DIR, "era5_monthly")
OUTPUT_DIR = os.path.join(BASE_DIR, "output_data_gridded")
GRIDDED_DIR = os.path.join(OUTPUT_DIR, "gridded")
PLOTS_DIR = os.path.join(OUTPUT_DIR, "plots")
STATS_DIR = os.path.join(OUTPUT_DIR, "statistics")

for d in [OUTPUT_DIR, GRIDDED_DIR, PLOTS_DIR, STATS_DIR, ERA5_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"📁 Base:    {BASE_DIR}")
print(f"📁 Ground:  {GT_DIR}")
print(f"📁 ERA5:    {ERA5_DIR}")
print(f"📁 Output:  {OUTPUT_DIR}")

In [17]:
# ── Variable Configuration ──
VARIABLE_CONFIG = {
    "AirTemperature": {
        "file": "USCRN_AirTemperature_2006_2021.xlsx",
        "convert": lambda x: x - 273.15,
        "era5_convert": lambda x: x - 273.15,
        "units": "°C",
        "long_name": "Near-Surface Air Temperature",
        "cmap": "RdYlBu_r",
        "clip_min": -80, "clip_max": 60,
    },
    "Precipitation": {
        "file": "USCRN_Precipitation_2006_2021.xlsx",
        "convert": lambda x: x,
        "era5_convert": lambda x: x * 1000 * 30,
        "units": "mm",
        "long_name": "Daily Precipitation",
        "cmap": "Blues",
        "clip_min": 0, "clip_max": 500,
    },
    "RelativeHumidity": {
        "file": "USCRN_RelativeHumidity_2006_2021.xlsx",
        "convert": lambda x: x,
        "era5_convert": lambda x: x,
        "units": "%",
        "long_name": "Relative Humidity",
        "cmap": "YlGnBu",
        "clip_min": 0, "clip_max": 100,
    },
    "SoilTemperature": {
        "file": "USCRN_SoilTemperature_2006_2021.xlsx",
        "convert": lambda x: x - 273.15,
        "era5_convert": lambda x: x - 273.15,
        "units": "°C",
        "long_name": "Soil Temperature (0-7 cm)",
        "cmap": "RdYlBu_r",
        "clip_min": -60, "clip_max": 70,
    },
    "SoilMoisture": {
        "file": "USCRN_SoilMoisture10cm_2006_2021.xlsx",
        "convert": lambda x: x,
        "era5_convert": lambda x: x,
        "units": "m³/m³",
        "long_name": "Volumetric Soil Water Content (10 cm)",
        "cmap": "YlGnBu",
        "clip_min": 0, "clip_max": 1.0,
    },
}

# ── Variogram models list ──
ALL_MODELS = ['spherical', 'exponential', 'gaussian', 'linear', 'circular']

# ── Best models per variable (from prior AIC analysis) ──
BEST_MODELS = {
    "AirTemperature":   "circular",
    "Precipitation":    "exponential",
    "RelativeHumidity": "circular",
    "SoilTemperature":  "circular",
    "SoilMoisture":     "spherical",
}

# ── Min stations for kriging ──
MIN_STATIONS = {"CONUS": 10, "Alaska": 5, "Hawaii": 1}

# ── Verify ──
print("✅ All configuration variables defined:")
print(f"   VARIABLE_CONFIG: {len(VARIABLE_CONFIG)} variables")
print(f"   ALL_MODELS:      {ALL_MODELS}")
print(f"   BEST_MODELS:     {len(BEST_MODELS)} entries")
print(f"   MIN_STATIONS:    {MIN_STATIONS}")
print(f"   Paths:           BASE_DIR, GT_DIR, OUTPUT_DIR, etc. all set")

In [18]:
# Variable configuration with unit conversions
VARIABLE_CONFIG = {
    "AirTemperature": {
        "file": "USCRN_AirTemperature_2006_2021.xlsx",
        "convert": lambda x: x - 273.15,         # K → °C
        "era5_convert": lambda x: x - 273.15,
        "units": "°C",
        "long_name": "Near-Surface Air Temperature",
        "cmap": "RdYlBu_r",
        "clip_min": -80, "clip_max": 60,
    },
    "Precipitation": {
        "file": "USCRN_Precipitation_2006_2021.xlsx",
        "convert": lambda x: x,                  # already mm
        "era5_convert": lambda x: x * 1000 * 30, # m/day → mm/month
        "units": "mm",
        "long_name": "Daily Precipitation",
        "cmap": "Blues",
        "clip_min": 0, "clip_max": 500,
    },
    "RelativeHumidity": {
        "file": "USCRN_RelativeHumidity_2006_2021.xlsx",
        "convert": lambda x: x,                  # already %
        "era5_convert": lambda x: x,
        "units": "%",
        "long_name": "Relative Humidity",
        "cmap": "YlGnBu",
        "clip_min": 0, "clip_max": 100,
    },
    "SoilTemperature": {
        "file": "USCRN_SoilTemperature_2006_2021.xlsx",
        "convert": lambda x: x - 273.15,
        "era5_convert": lambda x: x - 273.15,
        "units": "°C",
        "long_name": "Soil Temperature (0-7 cm)",
        "cmap": "RdYlBu_r",
        "clip_min": -60, "clip_max": 70,
    },
    "SoilMoisture": {
        "file": "USCRN_SoilMoisture10cm_2006_2021.xlsx",
        "convert": lambda x: x,
        "era5_convert": lambda x: x,
        "units": "m³/m³",
        "long_name": "Volumetric Soil Water Content (10 cm)",
        "cmap": "YlGnBu",
        "clip_min": 0, "clip_max": 1.0,
    },
}

ALL_MODELS = ['spherical', 'exponential', 'gaussian', 'linear', 'circular']
print(f"📊 {len(VARIABLE_CONFIG)} variables × {len(ALL_MODELS)} models")

---
# 2. Station Data Loading

Load USCRN station metadata and identify regions (CONUS / Alaska / Hawaii).

In [19]:
# Load station metadata
stations = pd.read_excel(os.path.join(GT_DIR, "All_CONUS_Station_Information_V2.xlsx"))

# Identify column for matching with data files
station_cols_sample = pd.read_excel(
    os.path.join(GT_DIR, "USCRN_AirTemperature_2006_2021.xlsx"), nrows=0
).columns[1:]

if any(str(c) in stations['StationName'].astype(str).values for c in station_cols_sample):
    STATION_KEY = 'StationName'
elif any(str(c) in stations['StationID'].astype(str).values for c in station_cols_sample):
    STATION_KEY = 'StationID'
    stations['StationID'] = stations['StationID'].astype(str)
else:
    STATION_KEY = 'StationName'

print(f"✅ Matching key: {STATION_KEY}")

# Remove non-US stations (e.g., SA_Tiksi in Siberia)
US_PREFIXES = ['AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA','HI','ID','IL',
               'IN','IA','KS','KY','LA','ME','MD','MA','MI','MN','MS','MO','MT',
               'NE','NV','NH','NJ','NM','NY','NC','ND','OH','OK','OR','PA','RI',
               'SC','SD','TN','TX','UT','VT','VA','WA','WV','WI','WY']

stations['Prefix'] = stations['StationName'].apply(lambda x: str(x).split('_')[0])
stations = stations[stations['Prefix'].isin(US_PREFIXES)].copy().reset_index(drop=True)

# Fix any positive longitudes (should all be negative for USA)
stations.loc[stations['Lon'] > 0, 'Lon'] = -stations.loc[stations['Lon'] > 0, 'Lon']

# Assign regions
stations['Region'] = 'CONUS'
stations.loc[stations['Prefix'] == 'AK', 'Region'] = 'Alaska'
stations.loc[stations['Prefix'] == 'HI', 'Region'] = 'Hawaii'

print(f"\n📍 Total US stations: {len(stations)}")
for r in ['CONUS', 'Alaska', 'Hawaii']:
    n = (stations['Region'] == r).sum()
    sub = stations[stations['Region'] == r]
    print(f"   {r:<8}: {n:>3} stations | "
          f"Lat [{sub['Lat'].min():.1f}, {sub['Lat'].max():.1f}] | "
          f"Lon [{sub['Lon'].min():.1f}, {sub['Lon'].max():.1f}]")

In [20]:
# Visualize station locations
fig = plt.figure(figsize=(18, 10))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
ax.set_extent([-180, -60, 15, 75], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.OCEAN, facecolor='#d4eaff')
ax.add_feature(cfeature.LAND, facecolor='#f5f5f5')
ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
ax.add_feature(cfeature.STATES, linewidth=0.3, edgecolor='gray')
ax.add_feature(cfeature.BORDERS, linewidth=0.5, linestyle='--')

colors = {'CONUS': 'red', 'Alaska': 'blue', 'Hawaii': 'green'}
for r, c in colors.items():
    sub = stations[stations['Region'] == r]
    ax.scatter(sub['Lon'], sub['Lat'], c=c, s=30, edgecolors='black',
               linewidth=0.3, label=f'{r} ({len(sub)})',
               transform=ccrs.PlateCarree(), zorder=5)

gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)
gl.top_labels = False; gl.right_labels = False
ax.legend(loc='lower left', fontsize=12)
ax.set_title(f'USCRN Station Network — {len(stations)} stations',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "station_map.svg"), dpi=200, bbox_inches='tight')
plt.show()

---
# 3. Region-Based Grid Setup

Create land-only 2° grids for CONUS and Alaska using US shapefiles. Hawaii uses 0.25° (see methodological justification in Section 6).

**Methodology Justification (Hawaii):**
- Only 2 USCRN stations available
- Islands < 2° grid cell width
- Kriging requires ≥5 stations for stable variogram (Webster & Oliver, 2007)
- IDW chosen following Longman et al. (2019) for sparse Hawaii networks

In [21]:
# Load US state shapefiles
shapefile_path = shpreader.natural_earth(
    resolution='50m', category='cultural', name='admin_1_states_provinces'
)
all_states = gpd.read_file(shapefile_path)
us_states = all_states[all_states['iso_a2'] == 'US'].copy()

conus_shape = us_states[~us_states['name'].isin(['Alaska', 'Hawaii'])].dissolve()
alaska_shape = us_states[us_states['name'] == 'Alaska'].dissolve()
hawaii_shape = us_states[us_states['name'] == 'Hawaii'].dissolve()

print(f"✅ Shapefiles: CONUS, Alaska, Hawaii loaded")

In [22]:
# ============================================================
# BUILD LAND-MASKED GRIDS — ALL REGIONS AT 2° (compliant)
# ============================================================
import numpy as np
from shapely.geometry import Point

def build_land_grid(shape_gdf, resolution=2.0, buffer=1.0,
                    proximity_fallback=True, fallback_distance=None):
    """
    Build a regular lat/lon grid masked to land cells of a region.
    
    Parameters
    ----------
    shape_gdf : GeoDataFrame
        Region shape (single geometry or union)
    resolution : float
        Grid resolution in degrees (default 2.0 per project instructions)
    buffer : float
        Padding around bounding box in degrees
    proximity_fallback : bool
        If True and no cells contain land, mark cells within `fallback_distance`
        of the geometry as land (handles small islands like Hawaii)
    fallback_distance : float or None
        Distance (degrees) for proximity fallback. Defaults to resolution.
    
    Returns
    -------
    dict with keys: full_lats, full_lons, land_mask, shape, bounds, resolution
    """
    bounds = shape_gdf.total_bounds  # [minx, miny, maxx, maxy]
    lon_min = np.floor(bounds[0] / resolution) * resolution - buffer
    lon_max = np.ceil(bounds[2] / resolution) * resolution + buffer
    lat_min = np.floor(bounds[1] / resolution) * resolution - buffer
    lat_max = np.ceil(bounds[3] / resolution) * resolution + buffer
    
    lons = np.arange(lon_min, lon_max + resolution, resolution)
    lats = np.arange(lat_min, lat_max + resolution, resolution)
    
    # Union of all geometries (in case multi-row gdf)
    geom = shape_gdf.geometry.unary_union
    
    # Standard land mask: cell center inside geometry OR within half-resolution
    land_mask = np.zeros((len(lats), len(lons)), dtype=bool)
    for i, lat in enumerate(lats):
        for j, lon in enumerate(lons):
            pt = Point(lon, lat)
            if geom.contains(pt) or geom.distance(pt) < resolution / 2:
                land_mask[i, j] = True
    
    # Proximity fallback for small regions (Hawaii at 2°)
    n_land = land_mask.sum()
    if n_land == 0 and proximity_fallback:
        fb_dist = fallback_distance if fallback_distance is not None else resolution
        print(f"     ⚠️ No land cells at {resolution}° using standard mask")
        print(f"     🔧 Applying proximity fallback (distance ≤ {fb_dist}°)")
        for i, lat in enumerate(lats):
            for j, lon in enumerate(lons):
                pt = Point(lon, lat)
                if geom.distance(pt) < fb_dist:
                    land_mask[i, j] = True
        n_land = land_mask.sum()
    
    return {
        'full_lats': lats,
        'full_lons': lons,
        'land_mask': land_mask,
        'shape': shape_gdf,
        'bounds': bounds,
        'resolution': resolution,
    }


# ============================================================
# Build all three regions at 2° (instruction-compliant)
# ============================================================
regions = {}
print("=" * 70)
print("BUILDING REGIONAL GRIDS @ 2° RESOLUTION (per project instructions)")
print("=" * 70)

# CONUS @ 2°
print(f"\n  📍 CONUS:")
regions['CONUS'] = build_land_grid(conus_shape, resolution=2.0, buffer=1.0)
print(f"     Grid:       {len(regions['CONUS']['full_lats'])} × "
      f"{len(regions['CONUS']['full_lons'])} = "
      f"{len(regions['CONUS']['full_lats']) * len(regions['CONUS']['full_lons'])} cells")
print(f"     Land cells: {regions['CONUS']['land_mask'].sum()}")
print(f"     Lat range:  {regions['CONUS']['full_lats'].min()}° to "
      f"{regions['CONUS']['full_lats'].max()}°")
print(f"     Lon range:  {regions['CONUS']['full_lons'].min()}° to "
      f"{regions['CONUS']['full_lons'].max()}°")

# Alaska @ 2°
print(f"\n  📍 Alaska:")
regions['Alaska'] = build_land_grid(alaska_shape, resolution=2.0, buffer=1.0)
print(f"     Grid:       {len(regions['Alaska']['full_lats'])} × "
      f"{len(regions['Alaska']['full_lons'])} = "
      f"{len(regions['Alaska']['full_lats']) * len(regions['Alaska']['full_lons'])} cells")
print(f"     Land cells: {regions['Alaska']['land_mask'].sum()}")
print(f"     Lat range:  {regions['Alaska']['full_lats'].min()}° to "
      f"{regions['Alaska']['full_lats'].max()}°")
print(f"     Lon range:  {regions['Alaska']['full_lons'].min()}° to "
      f"{regions['Alaska']['full_lons'].max()}°")

# Hawaii @ 2° (with proximity fallback for small islands)
print(f"\n  📍 Hawaii:")
regions['Hawaii'] = build_land_grid(
    hawaii_shape,
    resolution=2.0,
    buffer=1.0,
    proximity_fallback=True,
    fallback_distance=2.0,  # cells within 2° of any island
)
n_hawaii = regions['Hawaii']['land_mask'].sum()
print(f"     Grid:       {len(regions['Hawaii']['full_lats'])} × "
      f"{len(regions['Hawaii']['full_lons'])} = "
      f"{len(regions['Hawaii']['full_lats']) * len(regions['Hawaii']['full_lons'])} cells")
print(f"     Land cells: {n_hawaii}")
print(f"     Lat range:  {regions['Hawaii']['full_lats'].min()}° to "
      f"{regions['Hawaii']['full_lats'].max()}°")
print(f"     Lon range:  {regions['Hawaii']['full_lons'].min()}° to "
      f"{regions['Hawaii']['full_lons'].max()}°")

# Final station-based safety net for Hawaii
if n_hawaii == 0:
    print(f"\n     ⚠️ Geometric fallback returned 0 cells")
    print(f"     🔧 Using station-proximity mask as final fallback")
    hw_stations = stations[stations['Region'] == 'Hawaii']
    full_lats = regions['Hawaii']['full_lats']
    full_lons = regions['Hawaii']['full_lons']
    land_mask = np.zeros((len(full_lats), len(full_lons)), dtype=bool)
    for i, lat in enumerate(full_lats):
        for j, lon in enumerate(full_lons):
            for _, s in hw_stations.iterrows():
                if abs(lat - s['Lat']) <= 2.0 and abs(lon - s['Lon']) <= 2.0:
                    land_mask[i, j] = True
                    break
    regions['Hawaii']['land_mask'] = land_mask
    print(f"     Hawaii: {land_mask.sum()} land points (station-proximity)")

# ── Verification ──
print(f"\n{'=' * 70}")
print(f"VERIFICATION: All regions at 2° (instruction-compliant)")
print(f"{'=' * 70}\n")
print(f"  {'Region':<10} {'Resolution':<12} {'Grid':<14} {'Land Cells':<12}")
print(f"  {'─'*10} {'─'*12} {'─'*14} {'─'*12}")
for region_name, region_data in regions.items():
    res = region_data['resolution']
    nlat = len(region_data['full_lats'])
    nlon = len(region_data['full_lons'])
    nland = region_data['land_mask'].sum()
    
    if abs(res - 2.0) < 0.01:
        flag = "✅"
    else:
        flag = "❌ WRONG RES"
    
    print(f"  {region_name:<10} {res}°{'':<10} {nlat}×{nlon}{'':<8} {nland} {flag}")

# Sanity-check: all at 2°?
all_2deg = all(abs(r['resolution'] - 2.0) < 0.01 for r in regions.values())
all_have_land = all(r['land_mask'].sum() > 0 for r in regions.values())

if all_2deg and all_have_land:
    print(f"\n  ✅ ALL REGIONS COMPLIANT — proceed to interpolation functions")
elif not all_2deg:
    raise RuntimeError("Not all regions are at 2°!")
elif not all_have_land:
    raise RuntimeError("Some regions have 0 land cells — investigate!")

---
# 4. Variogram Model Selection — AIC + Cross-Validation

Test 5 variogram models per variable on sample days using:
- **AIC** (Akaike Information Criterion): goodness of fit
- **LOO Cross-Validation RMSE**: prediction accuracy

This produces an **internal** ranking. The **external** ranking (vs ERA5) is computed in Section 8.

In [23]:
# Variogram model functions
def spherical_model(h, psill, range_a, nugget):
    h = np.asarray(h, dtype=float)
    return np.where(h < range_a,
        nugget + psill * (1.5*(h/range_a) - 0.5*(h/range_a)**3),
        nugget + psill)

def exponential_model(h, psill, range_a, nugget):
    h = np.asarray(h, dtype=float)
    return nugget + psill * (1.0 - np.exp(-3.0 * h / range_a))

def gaussian_model(h, psill, range_a, nugget):
    h = np.asarray(h, dtype=float)
    return nugget + psill * (1.0 - np.exp(-3.0 * (h/range_a)**2))

def linear_model(h, slope, nugget):
    h = np.asarray(h, dtype=float)
    return nugget + slope * h

def circular_model(h, psill, range_a, nugget):
    h = np.asarray(h, dtype=float)
    gamma = np.zeros_like(h)
    mask = h < range_a
    if np.any(mask):
        h_ratio = np.clip(h[mask]/range_a, 0, 0.9999)
        gamma[mask] = nugget + psill * (
            1.0 - (2.0/np.pi) * (np.arccos(h_ratio) - h_ratio*np.sqrt(1-h_ratio**2)))
    gamma[~mask] = nugget + psill
    return gamma

# Custom circular for PyKrige
def circular_pykrige(m, d):
    return circular_model(d, *m)

VARIOGRAM_FIT = {
    'spherical':   {'func': spherical_model,   'n_params': 3},
    'exponential': {'func': exponential_model, 'n_params': 3},
    'gaussian':    {'func': gaussian_model,    'n_params': 3},
    'linear':      {'func': linear_model,      'n_params': 2},
    'circular':    {'func': circular_model,    'n_params': 3},
}

PYKRIGE_CONFIG = {
    'spherical':   {'name': 'spherical', 'custom': False},
    'exponential': {'name': 'exponential', 'custom': False},
    'gaussian':    {'name': 'gaussian', 'custom': False},
    'linear':      {'name': 'linear', 'custom': False},
    'circular':    {'name': circular_pykrige, 'custom': True},
}

def compute_variogram(lons, lats, values, n_lags=15):
    mask = ~np.isnan(values)
    if mask.sum() < 5:
        return None, None
    cl, ca, cv = lons[mask], lats[mask], values[mask]
    coords = np.column_stack([cl, ca])
    dists = squareform(pdist(coords))
    n = len(cv)
    diffs = np.zeros((n, n))
    for i in range(n):
        for j in range(i+1, n):
            diffs[i,j] = (cv[i]-cv[j])**2
            diffs[j,i] = diffs[i,j]
    max_d = np.max(dists)/2
    edges = np.linspace(0, max_d, n_lags+1)
    centers = (edges[:-1]+edges[1:])/2
    sv = np.zeros(n_lags); pairs = np.zeros(n_lags, dtype=int)
    for k in range(n_lags):
        m = np.triu((dists > edges[k]) & (dists <= edges[k+1]), k=1)
        if m.sum() > 0:
            sv[k] = np.sum(diffs[m]) / (2*m.sum())
            pairs[k] = m.sum()
    valid = pairs > 0
    return centers[valid], sv[valid]

def compute_aic(n, rss, k):
    if rss <= 0 or n <= k:
        return np.inf
    return n * np.log(rss/n) + 2*k

def fit_variogram(lags, sv, model_name):
    info = VARIOGRAM_FIT[model_name]
    func, k = info['func'], info['n_params']
    if len(lags) < k+1:
        return None
    data_var = np.nanmax(sv)
    max_lag = np.nanmax(lags)
    try:
        if model_name == 'linear':
            p0 = [data_var/max_lag, 0.0]
            bounds = ([0,0], [np.inf, data_var])
        else:
            p0 = [data_var*0.9, max_lag*0.5, data_var*0.1]
            bounds = ([0,0,0], [data_var*3, max_lag*2, data_var])
        popt, _ = curve_fit(func, lags, sv, p0=p0, bounds=bounds, maxfev=10000)
        fitted = func(lags, *popt)
        rss = np.sum((sv - fitted)**2)
        return {'aic': compute_aic(len(lags), rss, k),
                'r2': 1 - rss/np.sum((sv-np.mean(sv))**2) if np.sum((sv-np.mean(sv))**2) > 0 else 0}
    except:
        return None

print("✅ Variogram functions defined")

In [24]:
# ============================================================
# AIC ANALYSIS — UNIFIED FOR CONUS AND ALASKA
# ============================================================
# Tests all 5 variogram models per variable per region using AIC
# Hawaii excluded (only 2 stations — not enough for variogram fitting)
# ============================================================

import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("=" * 70)
print("AIC ANALYSIS — UNIFIED (CONUS + ALASKA, every 30th day)")
print("=" * 70)

# ── Configuration ──
ANALYSIS_REGIONS = ['CONUS', 'Alaska']
sample_step = 30  # days
MIN_STATIONS_AIC = {'CONUS': 10, 'Alaska': 5}

# Storage
all_aic_results = {}  # {region: {variable: {model: mean_aic}}}
all_aic_per_day = {}  # {region: {variable: {model: [aic1, aic2, ...]}}}
best_models_per_region = {}  # {region: {variable: best_model}}

start_time = time.time()

# ============================================================
# STEP 1: Run AIC for each region
# ============================================================
for region_name in ANALYSIS_REGIONS:
    region_st = stations[stations['Region'] == region_name]
    min_stations = MIN_STATIONS_AIC[region_name]
    
    print(f"\n{'█' * 70}")
    print(f"  📍 REGION: {region_name} ({len(region_st)} stations, "
          f"min={min_stations} required)")
    print(f"{'█' * 70}")
    
    region_results = {}
    region_per_day = {}
    
    for var_name, var_cfg in VARIABLE_CONFIG.items():
        print(f"\n  📊 {var_name}")
        
        # Load data
        df = pd.read_excel(os.path.join(GT_DIR, var_cfg['file']))
        date_col = df.columns[0]
        st_cols = df.columns[1:]
        df[st_cols] = df[st_cols].apply(pd.to_numeric, errors='coerce')
        df[st_cols] = var_cfg['convert'](df[st_cols])
        
        # Map stations
        v_lons, v_lats, v_cols = [], [], []
        for col in st_cols:
            m = region_st[region_st[STATION_KEY] == col]
            if len(m) > 0:
                v_lons.append(float(m['Lon'].values[0]))
                v_lats.append(float(m['Lat'].values[0]))
                v_cols.append(col)
        v_lons, v_lats = np.array(v_lons), np.array(v_lats)
        
        if len(v_cols) < min_stations:
            print(f"     ⚠️ Only {len(v_cols)} stations matched (need ≥{min_stations}) — skipping")
            continue
        
        # Sample days
        sample_days = list(range(0, len(df), sample_step))
        aic_scores = {mn: [] for mn in ALL_MODELS}
        
        for d_idx in tqdm(sample_days, desc=f"     {var_name}", leave=False):
            row = df.iloc[d_idx]
            vals = np.array([float(row[c]) if pd.notna(row[c]) else np.nan for c in v_cols])
            
            valid_n = (~np.isnan(vals)).sum()
            if valid_n < min_stations:
                continue
            
            lags, sv = compute_variogram(v_lons, v_lats, vals)
            if lags is None or len(lags) < 5:
                continue
            
            for mn in ALL_MODELS:
                r = fit_variogram(lags, sv, mn)
                if r is not None and np.isfinite(r['aic']):
                    aic_scores[mn].append(r['aic'])
        
        # Compute means
        means = {m: np.nanmean(s) if len(s) > 0 else np.inf 
                 for m, s in aic_scores.items()}
        valid_means = {m: v for m, v in means.items() if np.isfinite(v)}
        
        if not valid_means:
            print(f"     ⚠️ No valid AIC scores")
            continue
        
        sorted_models = sorted(valid_means.items(), key=lambda x: x[1])
        best = sorted_models[0]
        
        # Print ranking
        print(f"     Stations: {len(v_cols)} | Days analyzed: {len(sample_days)}")
        for rank, (mn, val) in enumerate(sorted_models, 1):
            marker = " ★" if rank == 1 else ""
            n_days = len(aic_scores[mn])
            print(f"       #{rank} {mn:<13} AIC={val:>8.2f}  ({n_days} days){marker}")
        
        region_results[var_name] = {m: round(v, 2) for m, v in means.items()}
        region_per_day[var_name] = aic_scores
    
    all_aic_results[region_name] = region_results
    all_aic_per_day[region_name] = region_per_day
    
    # Best model per variable for this region
    best_models_per_region[region_name] = {
        v: min({m: s for m, s in scores.items() if np.isfinite(s)}.items(),
               key=lambda x: x[1])[0]
        for v, scores in region_results.items()
        if any(np.isfinite(s) for s in scores.values())
    }

elapsed = (time.time() - start_time) / 60

# ============================================================
# STEP 2: Build comprehensive results table
# ============================================================
print(f"\n{'=' * 70}")
print("UNIFIED AIC RESULTS")
print(f"{'=' * 70}")

# Long-format table for CSV
unified_rows = []
for region in ANALYSIS_REGIONS:
    if region not in all_aic_results:
        continue
    for var_name in VARIABLE_CONFIG:
        if var_name not in all_aic_results[region]:
            continue
        scores = all_aic_results[region][var_name]
        sorted_m = sorted([(m, v) for m, v in scores.items() if np.isfinite(v)],
                          key=lambda x: x[1])
        for rank, (mn, val) in enumerate(sorted_m, 1):
            unified_rows.append({
                "Region": region,
                "Variable": var_name,
                "Model": mn,
                "Mean_AIC": val,
                "Rank": rank,
                "Is_Best": rank == 1,
                "N_Days": len(all_aic_per_day[region][var_name][mn]),
            })

unified_df = pd.DataFrame(unified_rows)
unified_df.to_csv(os.path.join(STATS_DIR, "aic_unified_results.csv"), index=False)
print(f"\n  ✅ Saved: aic_unified_results.csv ({len(unified_df)} rows)")

# Wide format (for easier reading)
wide_rows = []
for region in ANALYSIS_REGIONS:
    if region not in all_aic_results:
        continue
    for var_name in VARIABLE_CONFIG:
        if var_name not in all_aic_results[region]:
            continue
        row = {"Region": region, "Variable": var_name}
        for mn in ALL_MODELS:
            row[f"AIC_{mn}"] = all_aic_results[region][var_name].get(mn, np.nan)
        row["Best_Model"] = best_models_per_region[region].get(var_name, "—")
        wide_rows.append(row)

wide_df = pd.DataFrame(wide_rows)
wide_df.to_csv(os.path.join(STATS_DIR, "aic_unified_wide.csv"), index=False)

# Cross-region comparison table
print(f"\n  CROSS-REGION BEST MODEL COMPARISON:")
print(f"  {'Variable':<22} {'CONUS Best':<15} {'Alaska Best':<15} {'Agreement'}")
print(f"  {'─'*22} {'─'*15} {'─'*15} {'─'*12}")

agreements = []
for var_name in VARIABLE_CONFIG:
    conus_best = best_models_per_region.get('CONUS', {}).get(var_name, '—')
    alaska_best = best_models_per_region.get('Alaska', {}).get(var_name, '—')
    
    if conus_best != '—' and alaska_best != '—':
        agree = "✅ MATCH" if conus_best == alaska_best else "⚠️ DIFFER"
        agreements.append(conus_best == alaska_best)
    else:
        agree = "—"
    
    print(f"  {var_name:<22} {conus_best:<15} {alaska_best:<15} {agree}")

if agreements:
    n_agree = sum(agreements)
    print(f"\n  Agreement: {n_agree}/{len(agreements)} ({n_agree/len(agreements)*100:.0f}%)")

# Combined best (average rank across regions)
INITIAL_BEST_MODELS = {}
for var_name in VARIABLE_CONFIG:
    rank_scores = {mn: [] for mn in ALL_MODELS}
    for region in ANALYSIS_REGIONS:
        if region not in all_aic_results or var_name not in all_aic_results[region]:
            continue
        scores = all_aic_results[region][var_name]
        sorted_m = sorted([(m, v) for m, v in scores.items() if np.isfinite(v)],
                          key=lambda x: x[1])
        for rank, (mn, _) in enumerate(sorted_m, 1):
            rank_scores[mn].append(rank)
    
    avg_ranks = {m: np.mean(r) for m, r in rank_scores.items() if len(r) > 0}
    if avg_ranks:
        INITIAL_BEST_MODELS[var_name] = min(avg_ranks.items(), key=lambda x: x[1])[0]

print(f"\n  📌 INITIAL BEST MODELS (AIC-based, combined CONUS+Alaska):")
for v, m in INITIAL_BEST_MODELS.items():
    print(f"    {v:<22} → {m}")

print(f"\n  ⏱️  Total AIC analysis time: {elapsed:.1f} minutes")

# ============================================================
# STEP 3: VISUALIZATIONS (SVG format)
# ============================================================
print(f"\n{'=' * 70}")
print("GENERATING VISUALIZATIONS (SVG)")
print(f"{'=' * 70}")

# Color scheme
MODEL_COLORS = {
    'spherical':   '#1f77b4',
    'exponential': '#ff7f0e',
    'gaussian':    '#2ca02c',
    'linear':      '#d62728',
    'circular':    '#9467bd',
}

# ─────────────────────────────────────────────────────────────
# PLOT 1: Mean AIC Bar Chart — CONUS vs Alaska side-by-side
# ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(20, 7))

variables_in_data = [v for v in VARIABLE_CONFIG
                      if v in all_aic_results.get('CONUS', {})
                      or v in all_aic_results.get('Alaska', {})]
n_vars = len(variables_in_data)
x = np.arange(n_vars)
width = 0.15

for ax_idx, region in enumerate(ANALYSIS_REGIONS):
    ax = axes[ax_idx]
    
    if region not in all_aic_results:
        ax.text(0.5, 0.5, f'No data for {region}',
                ha='center', va='center', transform=ax.transAxes)
        continue
    
    for m_idx, model_name in enumerate(ALL_MODELS):
        vals = []
        for var_name in variables_in_data:
            scores = all_aic_results[region].get(var_name, {})
            v = scores.get(model_name, np.nan)
            vals.append(v if np.isfinite(v) else 0)
        
        offset = (m_idx - 2) * width
        bars = ax.bar(x + offset, vals, width,
                      label=model_name,
                      color=MODEL_COLORS[model_name],
                      edgecolor='black', alpha=0.85)
        
        # Mark best per variable
        for i, v in enumerate(vals):
            if v == 0:
                continue
            var_name = variables_in_data[i]
            scores = all_aic_results[region].get(var_name, {})
            valid_scores = {m: s for m, s in scores.items() if np.isfinite(s)}
            if valid_scores and abs(v - min(valid_scores.values())) < 1e-4:
                ax.text(x[i] + offset, v + max(vals) * 0.02,
                       '★', ha='center', fontsize=14, color='red',
                       fontweight='bold')
    
    ax.set_xticks(x)
    ax.set_xticklabels(variables_in_data, rotation=30, ha='right', fontsize=10)
    ax.set_ylabel('Mean AIC', fontsize=12)
    ax.set_title(f'{region} — AIC by Variogram Model\n(Lower = Better, ★ = Best)',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=9, loc='upper right')
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Variogram Model Selection via AIC — CONUS vs Alaska',
             fontsize=15, fontweight='bold', y=1.03)
plt.tight_layout()

svg_path = os.path.join(PLOTS_DIR, "aic_comparison_regions.svg")
png_path = os.path.join(PLOTS_DIR, "aic_comparison_regions.png")
plt.savefig(svg_path, format='svg', bbox_inches='tight')
plt.savefig(png_path, dpi=200, bbox_inches='tight')
plt.show()
print(f"  ✅ {os.path.basename(svg_path)} (and PNG)")

# ─────────────────────────────────────────────────────────────
# PLOT 2: Heatmap — AIC values per Region per Variable per Model
# ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(20, 6))

for ax_idx, region in enumerate(ANALYSIS_REGIONS):
    ax = axes[ax_idx]
    
    if region not in all_aic_results:
        ax.text(0.5, 0.5, f'No data for {region}',
                ha='center', va='center', transform=ax.transAxes)
        continue
    
    heatmap = np.full((len(ALL_MODELS), n_vars), np.nan)
    for m_idx, model_name in enumerate(ALL_MODELS):
        for v_idx, var_name in enumerate(variables_in_data):
            scores = all_aic_results[region].get(var_name, {})
            v = scores.get(model_name, np.nan)
            if np.isfinite(v):
                heatmap[m_idx, v_idx] = v
    
    valid_vals = heatmap[~np.isnan(heatmap)]
    if len(valid_vals) == 0:
        continue
    
    im = ax.imshow(heatmap, cmap='YlOrRd', aspect='auto')
    plt.colorbar(im, ax=ax, label='AIC')
    
    ax.set_xticks(np.arange(n_vars))
    ax.set_xticklabels(variables_in_data, rotation=30, ha='right', fontsize=10)
    ax.set_yticks(np.arange(len(ALL_MODELS)))
    ax.set_yticklabels(ALL_MODELS, fontsize=10)
    
    thresh = np.nanpercentile(valid_vals, 60) if len(valid_vals) > 1 else 0
    
    for i in range(len(ALL_MODELS)):
        for j in range(n_vars):
            if not np.isnan(heatmap[i, j]):
                col_min = np.nanmin(heatmap[:, j])
                weight = 'bold' if abs(heatmap[i, j] - col_min) < 1e-4 else 'normal'
                color = 'white' if heatmap[i, j] > thresh else 'black'
                # Add ★ for best
                txt = f'{heatmap[i,j]:.2f}'
                if abs(heatmap[i, j] - col_min) < 1e-4:
                    txt += ' ★'
                ax.text(j, i, txt, ha='center', va='center',
                       fontsize=9, fontweight=weight, color=color)
    
    ax.set_title(f'{region} — AIC Heatmap (Bold + ★ = Best per Variable)',
                 fontsize=13, fontweight='bold')

plt.suptitle('AIC Heatmap: Models × Variables × Regions',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()

svg_path = os.path.join(PLOTS_DIR, "aic_heatmap_regions.svg")
png_path = os.path.join(PLOTS_DIR, "aic_heatmap_regions.png")
plt.savefig(svg_path, format='svg', bbox_inches='tight')
plt.savefig(png_path, dpi=200, bbox_inches='tight')
plt.show()
print(f"  ✅ {os.path.basename(svg_path)} (and PNG)")

# ─────────────────────────────────────────────────────────────
# PLOT 3: Box plots of AIC distribution per model per variable
# ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 5, figsize=(28, 12))

for r_idx, region in enumerate(ANALYSIS_REGIONS):
    if region not in all_aic_per_day:
        continue
    
    for v_idx, var_name in enumerate(variables_in_data[:5]):
        ax = axes[r_idx, v_idx]
        
        if var_name not in all_aic_per_day[region]:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center',
                    transform=ax.transAxes)
            ax.set_title(f'{region}: {var_name}', fontsize=10)
            continue
        
        per_day = all_aic_per_day[region][var_name]
        box_data = []
        labels = []
        colors_list = []
        
        for mn in ALL_MODELS:
            scores = per_day.get(mn, [])
            scores = [s for s in scores if np.isfinite(s)]
            if len(scores) > 0:
                # Cap extreme outliers for visualization
                scores_array = np.array(scores)
                p99 = np.percentile(scores_array, 99) if len(scores_array) > 5 else max(scores_array)
                scores_clipped = scores_array[scores_array <= p99]
                box_data.append(scores_clipped)
                labels.append(mn)
                colors_list.append(MODEL_COLORS[mn])
        
        if box_data:
            bp = ax.boxplot(box_data, labels=labels, patch_artist=True, widths=0.6)
            for patch, color in zip(bp['boxes'], colors_list):
                patch.set_facecolor(color)
                patch.set_alpha(0.6)
            
            # Mark best with star
            best_for_this = best_models_per_region.get(region, {}).get(var_name, '')
            if best_for_this in labels:
                idx = labels.index(best_for_this)
                ax.plot(idx + 1, ax.get_ylim()[1] * 0.95, '*',
                       markersize=20, color='gold',
                       markeredgecolor='red', markeredgewidth=1.5,
                       zorder=10)
        
        ax.set_title(f'{region}: {var_name}', fontsize=11, fontweight='bold')
        ax.set_ylabel('AIC' if v_idx == 0 else '')
        ax.tick_params(axis='x', rotation=30, labelsize=9)
        ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('AIC Distribution per Model — Box Plots (★ = Best)',
             fontsize=15, fontweight='bold', y=1.005)
plt.tight_layout()

svg_path = os.path.join(PLOTS_DIR, "aic_boxplots_regions.svg")
png_path = os.path.join(PLOTS_DIR, "aic_boxplots_regions.png")
plt.savefig(svg_path, format='svg', bbox_inches='tight')
plt.savefig(png_path, dpi=200, bbox_inches='tight')
plt.show()
print(f"  ✅ {os.path.basename(svg_path)} (and PNG)")

# ─────────────────────────────────────────────────────────────
# PLOT 4: Best Model Summary Table (visual)
# ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))
ax.axis('off')

# Build table data
table_rows = []
for var_name in VARIABLE_CONFIG:
    conus_best = best_models_per_region.get('CONUS', {}).get(var_name, '—')
    alaska_best = best_models_per_region.get('Alaska', {}).get(var_name, '—')
    combined_best = INITIAL_BEST_MODELS.get(var_name, '—')
    
    if conus_best != '—' and alaska_best != '—':
        agreement = "✓ Match" if conus_best == alaska_best else "✗ Differ"
    else:
        agreement = "—"
    
    # Get AIC values
    conus_aic = all_aic_results.get('CONUS', {}).get(var_name, {}).get(conus_best, np.nan)
    alaska_aic = all_aic_results.get('Alaska', {}).get(var_name, {}).get(alaska_best, np.nan)
    
    table_rows.append([
        var_name,
        f"{conus_best}\n(AIC={conus_aic:.2f})" if np.isfinite(conus_aic) else conus_best,
        f"{alaska_best}\n(AIC={alaska_aic:.2f})" if np.isfinite(alaska_aic) else alaska_best,
        combined_best,
        agreement,
    ])

col_labels = ['Variable', 'CONUS\n(207 stations)', 'Alaska\n(23 stations)',
              'Combined\nBest', 'Cross-Region\nAgreement']

table = ax.table(cellText=table_rows, colLabels=col_labels,
                cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 2.2)

# Style header
for j in range(len(col_labels)):
    table[0, j].set_facecolor('#2F5496')
    table[0, j].set_text_props(color='white', fontweight='bold')

# Color rows by agreement
for i in range(1, len(table_rows) + 1):
    agreement_str = table_rows[i-1][4]
    if 'Match' in agreement_str:
        for j in range(len(col_labels)):
            table[i, j].set_facecolor('#E8F5E9')
    elif 'Differ' in agreement_str:
        for j in range(len(col_labels)):
            table[i, j].set_facecolor('#FFF3E0')

ax.set_title('Variogram Model Selection Summary (AIC-based)\n'
             'Cross-Region Best Model Agreement',
             fontsize=14, fontweight='bold', pad=20)

svg_path = os.path.join(PLOTS_DIR, "aic_best_model_summary.svg")
png_path = os.path.join(PLOTS_DIR, "aic_best_model_summary.png")
plt.savefig(svg_path, format='svg', bbox_inches='tight')
plt.savefig(png_path, dpi=200, bbox_inches='tight')
plt.show()
print(f"  ✅ {os.path.basename(svg_path)} (and PNG)")

# ============================================================
# FINAL SUMMARY
# ============================================================
print(f"\n{'=' * 70}")
print("UNIFIED AIC ANALYSIS — COMPLETE")
print(f"{'=' * 70}")

print(f"""
  ⏱️  Total time:    {elapsed:.1f} minutes
  📊  Regions:       {len(all_aic_results)} ({', '.join(all_aic_results.keys())})
  📊  Variables:     {len(variables_in_data)}
  📊  Models tested: {len(ALL_MODELS)}

  📁 OUTPUTS:
  ├── aic_unified_results.csv      (long format)
  ├── aic_unified_wide.csv         (wide format)
  ├── aic_comparison_regions.svg   (bar chart per region)
  ├── aic_heatmap_regions.svg      (model × variable heatmaps)
  ├── aic_boxplots_regions.svg     (AIC distributions)
  └── aic_best_model_summary.svg   (summary table)

  🎯 INITIAL BEST MODELS (use these for Section 5):
""")

for v, m in INITIAL_BEST_MODELS.items():
    print(f"     {v:<22} → {m}")


In [25]:
# ============================================================
# DIAGNOSTIC: Why does Alaska SoilMoisture fail?
# ============================================================
import os
import numpy as np
import pandas as pd

print("=" * 70)
print("DIAGNOSTIC: Alaska SoilMoisture data availability")
print("=" * 70)

var_cfg = VARIABLE_CONFIG['SoilMoisture']
df = pd.read_excel(os.path.join(GT_DIR, var_cfg['file']))
date_col = df.columns[0]
st_cols = df.columns[1:]

# Convert to numeric
df[st_cols] = df[st_cols].apply(pd.to_numeric, errors='coerce')
df[st_cols] = var_cfg['convert'](df[st_cols])

# Map to Alaska stations
alaska_st = stations[stations['Region'] == 'Alaska']
ak_cols = []
for col in st_cols:
    if (alaska_st[STATION_KEY] == col).any():
        ak_cols.append(col)

print(f"\n  Alaska stations in registry: {len(alaska_st)}")
print(f"  Alaska SM columns matched in Excel: {len(ak_cols)}")

if len(ak_cols) == 0:
    print(f"\n  ❌ CRITICAL: NO Alaska stations found in SoilMoisture file!")
    print(f"     This is a column-naming mismatch issue.")
    print(f"\n  First 10 Alaska station keys:")
    for s in alaska_st[STATION_KEY].head(10):
        print(f"     {s}")
    print(f"\n  First 10 columns in SoilMoisture file:")
    for c in list(st_cols)[:10]:
        print(f"     {c}")
else:
    # Per-day valid count
    valid_per_day = df[ak_cols].notna().sum(axis=1)
    
    print(f"\n  Daily valid station counts (out of {len(ak_cols)}):")
    print(f"     Min:    {valid_per_day.min()}")
    print(f"     Max:    {valid_per_day.max()}")
    print(f"     Mean:   {valid_per_day.mean():.1f}")
    print(f"     Median: {valid_per_day.median():.0f}")
    
    print(f"\n  Days with >= N valid stations:")
    for n in [1, 3, 5, 8, 10, 15]:
        cnt = (valid_per_day >= n).sum()
        pct = 100 * cnt / len(valid_per_day)
        marker = " ← MIN_STATIONS threshold" if n == 5 else ""
        print(f"     >= {n:>2}: {cnt:>5} days ({pct:>5.1f}%){marker}")
    
    # Per-station availability
    print(f"\n  Per-station data availability:")
    for col in ak_cols:
        n_valid = df[col].notna().sum()
        pct = 100 * n_valid / len(df)
        flag = " ⚠️" if pct < 30 else ""
        print(f"     {col}: {n_valid:>5} days ({pct:>5.1f}%){flag}")
    
    # Sampled days analysis (every 30th)
    sampled_idx = list(range(0, len(df), 30))
    sampled = df.iloc[sampled_idx]
    sampled_valid = sampled[ak_cols].notna().sum(axis=1)
    
    print(f"\n  AIC sampled days (every 30th, n={len(sampled)}):")
    print(f"     Days with >= 5 valid: {(sampled_valid >= 5).sum()}")
    print(f"     Days with >= 3 valid: {(sampled_valid >= 3).sum()}")
    print(f"     Days with valid: {(sampled_valid >= 1).sum()}")

print(f"\n{'=' * 70}")

## INTERPOLATION FUNCTIONS

In [26]:
# ============================================================
# CELL: INTERPOLATION FUNCTIONS (krige_day, idw_interp, simple_avg)
# ============================================================

# - krige_day:   Ordinary Kriging for one timestep
# - idw_interp:  Inverse Distance Weighting (Hawaii fallback)
# - simple_avg:  Constant-field fallback when kriging/IDW fail
# ============================================================

import numpy as np
import warnings
warnings.filterwarnings('ignore')

try:
    from pykrige.ok import OrdinaryKriging
    print("  ✅ PyKrige available")
except ImportError:
    raise ImportError("PyKrige required. Install with: pip install pykrige")


def krige_day(lons, lats, vals, region_data, model='spherical',
              nlags=10, enable_plotting=False):
    """
    Ordinary Kriging interpolation for one day's station observations.
    
    Parameters
    ----------
    lons, lats : 1D arrays
        Station longitudes and latitudes (degrees)
    vals : 1D array
        Station observations (may contain NaN)
    region_data : dict
        Must contain 'full_lats', 'full_lons', 'land_mask'
    model : str
        Variogram model: 'spherical', 'exponential', 'gaussian',
        'linear', or 'circular'
    nlags : int
        Number of variogram lag bins
    
    Returns
    -------
    z : 2D array (n_lat, n_lon)
        Kriged values; NaN over ocean cells
    ss : 2D array (n_lat, n_lon)
        Kriging variance; NaN over ocean cells
    """
    # Filter NaNs
    valid = ~np.isnan(vals)
    if valid.sum() < 5:  # need ≥5 points for variogram fitting
        return _empty_grid(region_data)
    
    lons_v = lons[valid]
    lats_v = lats[valid]
    vals_v = vals[valid]
    
    # Remove duplicate locations (PyKrige fails on duplicates)
    coords = np.column_stack([lons_v, lats_v])
    _, unique_idx = np.unique(coords, axis=0, return_index=True)
    if len(unique_idx) < 5:
        return _empty_grid(region_data)
    lons_v = lons_v[unique_idx]
    lats_v = lats_v[unique_idx]
    vals_v = vals_v[unique_idx]
    
    # Check for zero-variance (all values identical → kriging fails)
    if np.std(vals_v) < 1e-10:
        z = np.full((len(region_data['full_lats']),
                     len(region_data['full_lons'])),
                    vals_v[0], dtype=np.float32)
        z[~region_data['land_mask']] = np.nan
        ss = np.zeros_like(z)
        ss[~region_data['land_mask']] = np.nan
        return z, ss
    
    try:
        ok = OrdinaryKriging(
            lons_v, lats_v, vals_v,
            variogram_model=model,
            nlags=min(nlags, max(3, len(lons_v) // 3)),
            enable_plotting=enable_plotting,
            verbose=False,
            coordinates_type='geographic',
        )
        z, ss = ok.execute(
            'grid',
            region_data['full_lons'],
            region_data['full_lats'],
        )
        z = np.asarray(z, dtype=np.float32)
        ss = np.asarray(ss, dtype=np.float32)
        # Mask ocean cells
        z[~region_data['land_mask']] = np.nan
        ss[~region_data['land_mask']] = np.nan
        return z, ss
    except Exception:
        return _empty_grid(region_data)


def idw_interp(lons, lats, vals, region_data, power=2, max_dist=None):
    """
    Inverse Distance Weighting interpolation for one day.
    Used for Hawaii where stations are too few for kriging.
    
    Parameters
    ----------
    lons, lats : 1D arrays
        Station coordinates
    vals : 1D array
        Station observations
    region_data : dict
        Must contain 'full_lats', 'full_lons', 'land_mask'
    power : float
        Distance weighting exponent (2 = standard IDW)
    max_dist : float or None
        If not None, cells > max_dist degrees from any station = NaN
    
    Returns
    -------
    z : 2D array (n_lat, n_lon)
    """
    valid = ~np.isnan(vals)
    if valid.sum() == 0:
        return np.full((len(region_data['full_lats']),
                        len(region_data['full_lons'])),
                       np.nan, dtype=np.float32)
    
    lons_v = lons[valid]
    lats_v = lats[valid]
    vals_v = vals[valid]
    
    # Build target grid
    grid_lon, grid_lat = np.meshgrid(region_data['full_lons'],
                                      region_data['full_lats'])
    
    # Compute distances (degrees, simple Euclidean)
    z = np.full(grid_lon.shape, np.nan, dtype=np.float32)
    
    for i in range(grid_lon.shape[0]):
        for j in range(grid_lon.shape[1]):
            if not region_data['land_mask'][i, j]:
                continue
            
            dx = lons_v - grid_lon[i, j]
            dy = lats_v - grid_lat[i, j]
            dist = np.sqrt(dx ** 2 + dy ** 2)
            
            # Exact match → use that value directly
            if (dist < 1e-9).any():
                z[i, j] = vals_v[dist < 1e-9][0]
                continue
            
            # Optional max-distance cutoff
            if max_dist is not None and (dist > max_dist).all():
                continue
            
            weights = 1.0 / (dist ** power)
            z[i, j] = np.sum(weights * vals_v) / np.sum(weights)
    
    return z


def simple_avg(vals, region_data):
    """
    Fallback: assign the mean of valid station values to all land cells.
    Used when both kriging and IDW are infeasible.
    
    Parameters
    ----------
    vals : 1D array
        Station observations (may contain NaN)
    region_data : dict
        Must contain 'land_mask', 'full_lats', 'full_lons'
    
    Returns
    -------
    z : 2D array (n_lat, n_lon) — constant value over land
    ss : 2D array (n_lat, n_lon) — variance estimate (sample variance)
    """
    valid = ~np.isnan(vals)
    if valid.sum() == 0:
        return _empty_grid(region_data)
    
    mean_val = np.mean(vals[valid])
    var_val = np.var(vals[valid]) if valid.sum() > 1 else 0.0
    
    n_lat = len(region_data['full_lats'])
    n_lon = len(region_data['full_lons'])
    z = np.full((n_lat, n_lon), mean_val, dtype=np.float32)
    ss = np.full((n_lat, n_lon), var_val, dtype=np.float32)
    
    z[~region_data['land_mask']] = np.nan
    ss[~region_data['land_mask']] = np.nan
    return z, ss


def _empty_grid(region_data):
    """Return NaN-filled grids matching region shape."""
    n_lat = len(region_data['full_lats'])
    n_lon = len(region_data['full_lons'])
    z = np.full((n_lat, n_lon), np.nan, dtype=np.float32)
    ss = np.full((n_lat, n_lon), np.nan, dtype=np.float32)
    return z, ss


# ── Verify functions are now defined ──
print("\n  ✅ Functions defined:")
print(f"     krige_day:   {krige_day.__doc__.strip().split(chr(10))[0]}")
print(f"     idw_interp:  {idw_interp.__doc__.strip().split(chr(10))[0]}")
print(f"     simple_avg:  {simple_avg.__doc__.strip().split(chr(10))[0]}")

# ── Quick sanity test ──
print("\n  🧪 Running quick sanity test...")
import numpy as np
test_region = {
    'full_lats': np.array([20.0, 21.0, 22.0]),
    'full_lons': np.array([-160.0, -159.0, -158.0]),
    'land_mask': np.ones((3, 3), dtype=bool),
}
test_lons = np.array([-159.5, -158.5])
test_lats = np.array([20.5, 21.5])
test_vals = np.array([10.0, 20.0])

z_idw = idw_interp(test_lons, test_lats, test_vals, test_region)
z_avg, ss_avg = simple_avg(test_vals, test_region)

print(f"     IDW test:    shape={z_idw.shape}, range=[{np.nanmin(z_idw):.2f}, {np.nanmax(z_idw):.2f}]")
print(f"     simple_avg:  shape={z_avg.shape}, value={np.nanmean(z_avg):.2f}")
print(f"\n  ✅ All interpolation functions ready — proceed to Section 5a")

### SECTION 5a: GENERATE DAILY GRIDDED DATA (TOP-2 MODELS)

In [27]:
# ============================================================
# CELL 2 — SECTION 5a: DAILY GRIDDED DATA GENERATION
# AIC-justified model pairs (Top-1 from AIC + Spherical anchor)
# ============================================================
import os, time, json
import numpy as np
import pandas as pd
import xarray as xr
from datetime import datetime
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("=" * 70)
print("SECTION 5a: DAILY GRIDDED DATA (AIC-justified model pairs)")
print("=" * 70)

# ── Pre-flight ──
required = ['VARIABLE_CONFIG','MIN_STATIONS','regions','stations',
            'STATION_KEY','GT_DIR','GRIDDED_DIR','STATS_DIR',
            'krige_day','idw_interp','simple_avg']
g = globals()
missing = [v for v in required if v not in g]
if missing:
    raise RuntimeError(f"Missing: {missing}")
print(f"  ✅ All {len(required)} required variables present")

hw_step = abs(regions['Hawaii']['full_lats'][1] - regions['Hawaii']['full_lats'][0])
if abs(hw_step - 2.0) > 0.01:
    raise RuntimeError(f"Hawaii grid is {hw_step}°, requires 2°")
print(f"  ✅ Hawaii grid verified at 2°")

# ── Exclusions ──
SKIP_COMBINATIONS = {('Alaska', 'SoilMoisture')}
print(f"\n  ⚠️ Excluded {len(SKIP_COMBINATIONS)} combinations:")
for r, v in SKIP_COMBINATIONS:
    print(f"     • {r} / {v}")

# ============================================================
# AIC-JUSTIFIED MODEL SELECTION
# Strategy: AIC Top-1 + Spherical anchor (replaces unstable circular)
# ============================================================
TOP_MODELS_PER_VAR = {
    'AirTemperature':   ['gaussian',    'spherical'],
    'Precipitation':    ['gaussian',    'spherical'],
    'RelativeHumidity': ['spherical',   'gaussian'],   # AIC#1=circular unstable
    'SoilTemperature':  ['gaussian',    'spherical'],
    'SoilMoisture':     ['exponential', 'spherical'],
}

AIC_JUSTIFICATION = {
    'AirTemperature': {
        'aic_top3': 'gaussian (42.64) | circular (44.12) | spherical (44.77)',
        'delta_aic_1to3': 2.13,
        'reasoning': 'gaussian=AIC#1; spherical replaces unstable circular (ΔAIC<3, equivalent)',
    },
    'Precipitation': {
        'aic_top3': 'gaussian (58.07) | circular (58.54) | spherical (58.69)',
        'delta_aic_1to3': 0.62,
        'reasoning': 'top-3 statistically tied; gaussian + spherical anchor',
    },
    'RelativeHumidity': {
        'aic_top3': 'circular (104.71) | spherical (105.07) | gaussian (106.13)',
        'delta_aic_1to3': 1.42,
        'reasoning': 'AIC#1=circular unstable; #2 (spherical) and #3 (gaussian) selected',
    },
    'SoilTemperature': {
        'aic_top3': 'gaussian (44.16) | circular (46.57) | spherical (47.28)',
        'delta_aic_1to3': 3.12,
        'reasoning': 'gaussian preferred; spherical = stable runner-up',
    },
    'SoilMoisture': {
        'aic_top3': 'exponential (-194.64) | spherical (-194.62) | circular (-194.53)',
        'delta_aic_1to3': 0.11,
        'reasoning': 'top-3 statistically identical; AIC#1 (exponential) + AIC#2 (spherical)',
    },
}

print(f"\n  📊 AIC-JUSTIFIED MODEL PAIRS:")
print(f"  {'Variable':<22} {'Pair':<28} {'ΔAIC #1↔#3':<12} Reasoning")
print(f"  {'─'*22} {'─'*28} {'─'*12} {'─'*45}")
for v, m in TOP_MODELS_PER_VAR.items():
    info = AIC_JUSTIFICATION[v]
    pair_str = f"{m[0]} + {m[1]}"
    print(f"  {v:<22} {pair_str:<28} {info['delta_aic_1to3']:<12.2f} {info['reasoning'][:45]}")

# Save justification for report
with open(os.path.join(STATS_DIR, "aic_justification.json"), 'w') as f:
    json.dump({
        'model_pairs': {v: list(m) for v, m in TOP_MODELS_PER_VAR.items()},
        'aic_justification': AIC_JUSTIFICATION,
        'methodology_summary': (
            'Two-stage variogram selection: (1) AIC top-1 from screening; '
            '(2) Spherical anchor (replaces unstable circular for n=207). '
            'Final winner per (region × variable) selected by ERA5 RMSE.'
        ),
    }, f, indent=2)

# ============================================================
# Spatial-variance validation (catches silent kriging failures)
# ============================================================
def validate_kriging_output(z, threshold=1e-4):
    valid = z[~np.isnan(z)]
    if len(valid) < 2: return False
    return float(np.ptp(valid)) > threshold

# ============================================================
# Main generation loop
# ============================================================
start_total = time.time()
gen_log = []
skipped_log = []

for region_name in ['CONUS', 'Alaska', 'Hawaii']:
    region_data = regions[region_name]
    region_st = stations[stations['Region'] == region_name]
    use_idw = (region_name == 'Hawaii')
    
    print(f"\n{'█'*70}")
    print(f"  📍 {region_name} | {len(region_st)} stations | "
          f"{region_data['land_mask'].sum()} land cells | "
          f"{'IDW' if use_idw else 'Kriging (AIC-justified pairs)'}")
    print(f"{'█'*70}")
    
    if region_data['land_mask'].sum() == 0:
        print(f"  ⚠️ No land cells — skipping"); continue
    
    for var_name, var_cfg in VARIABLE_CONFIG.items():
        if (region_name, var_name) in SKIP_COMBINATIONS:
            print(f"\n  ⚠️ SKIP {var_name}: in exclusion list")
            skipped_log.append({"region": region_name, "variable": var_name,
                                "reason": "Only 1 active station (AK_Kenai_29_ENE)"})
            continue
        
        models_to_run = ['idw'] if use_idw else TOP_MODELS_PER_VAR[var_name]
        
        for model in models_to_run:
            t_start = time.time()
            
            if use_idw:
                output_file = os.path.join(GRIDDED_DIR,
                    f"Gridded_{var_name}_{region_name}_idw_2deg_daily.nc")
                method_label = "IDW"
            else:
                output_file = os.path.join(GRIDDED_DIR,
                    f"Gridded_{var_name}_{region_name}_{model}_2deg_daily.nc")
                method_label = f"OK ({model})"
            
            # Skip-if-exists with quality check
            if os.path.exists(output_file) and os.path.getsize(output_file) > 100*1024:
                try:
                    ds_chk = xr.open_dataset(output_file)
                    data_chk = ds_chk[var_name].values
                    pdv = np.nanvar(data_chk.reshape(data_chk.shape[0], -1), axis=1)
                    med_var = float(np.nanmedian(pdv))
                    n_avg_a = ds_chk.attrs.get('n_days_averaged', 0)
                    n_krig_a = ds_chk.attrs.get('n_days_kriged', 0)
                    ds_chk.close()
                    is_fb = (med_var < 1e-4) or (n_avg_a > 0.8*(n_avg_a+n_krig_a))
                    
                    if is_fb:
                        print(f"\n  ⚠️ {var_name} [{model}]: existing is FALLBACK — re-running")
                        os.remove(output_file)
                    else:
                        sz = os.path.getsize(output_file)/(1024**2)
                        print(f"\n  ⏭ {var_name} [{model}]: SKIP ({sz:.1f} MB, real)")
                        gen_log.append({
                            "region": region_name, "variable": var_name, "model": model,
                            "method": method_label, "size_mb": round(sz,1),
                            "median_spatial_var": round(med_var,4),
                            "file": output_file, "skipped": True,
                            "is_fallback": False, "quality": "real_kriging"})
                        continue
                except Exception:
                    if os.path.exists(output_file): os.remove(output_file)
            
            print(f"\n  📊 {var_name} ({var_cfg['units']}) — {method_label}")
            
            # Load data
            try:
                df = pd.read_excel(os.path.join(GT_DIR, var_cfg['file']))
                date_col = df.columns[0]
                st_cols = df.columns[1:]
                df[st_cols] = df[st_cols].apply(pd.to_numeric, errors='coerce')
                df[st_cols] = var_cfg['convert'](df[st_cols])
            except Exception as e:
                print(f"     ❌ Load failed: {e}"); continue
            
            # Match stations
            v_lons, v_lats, v_cols = [], [], []
            for col in st_cols:
                m = region_st[region_st[STATION_KEY] == col]
                if len(m) > 0:
                    v_lons.append(float(m['Lon'].values[0]))
                    v_lats.append(float(m['Lat'].values[0]))
                    v_cols.append(col)
            v_lons, v_lats = np.array(v_lons), np.array(v_lats)
            
            if len(v_cols) == 0:
                print(f"     ⚠️ No stations matched"); continue
            print(f"     Stations matched: {len(v_cols)}")
            
            valid_per_day = df[v_cols].notna().sum(axis=1)
            min_st = MIN_STATIONS[region_name]
            n_usable = ((valid_per_day >= min_st).sum() if not use_idw 
                        else (valid_per_day >= 1).sum())
            print(f"     Usable days: {n_usable}/{len(df)} ({100*n_usable/len(df):.1f}%)")
            
            if n_usable < 30:
                print(f"     ⚠️ Too few usable days — skipping")
                skipped_log.append({"region": region_name, "variable": var_name,
                                    "reason": f"Only {n_usable} usable days"})
                continue
            
            dates = pd.to_datetime(df[date_col])
            n_days = len(dates)
            n_lat = len(region_data['full_lats'])
            n_lon = len(region_data['full_lons'])
            
            gridded = np.full((n_days, n_lat, n_lon), np.nan, dtype=np.float32)
            variance = np.full((n_days, n_lat, n_lon), np.nan, dtype=np.float32)
            n_ok = n_avg = n_skip = n_error = 0
            
            for t in tqdm(range(n_days), desc=f"     Days", leave=False):
                try:
                    row = df.iloc[t]
                    vals = np.array([float(row[c]) if pd.notna(row[c]) else np.nan
                                     for c in v_cols])
                    valid = (~np.isnan(vals)).sum()
                    if valid < 1: n_skip += 1; continue
                    
                    if use_idw:
                        z = idw_interp(v_lons, v_lats, vals, region_data)
                        ss = np.zeros_like(z)
                        if not np.all(np.isnan(z)): n_ok += 1
                        else: n_skip += 1; continue
                    elif valid >= min_st and valid >= 5:
                        z, ss = krige_day(v_lons, v_lats, vals, region_data, model)
                        if np.all(np.isnan(z)) or not validate_kriging_output(z):
                            z, ss = simple_avg(vals, region_data); n_avg += 1
                        else:
                            n_ok += 1
                    else:
                        z, ss = simple_avg(vals, region_data); n_avg += 1
                    
                    gridded[t] = z
                    variance[t] = ss
                except Exception:
                    n_error += 1
            
            gridded = np.clip(gridded, var_cfg['clip_min'], var_cfg['clip_max'])
            
            land_vals = gridded[:, region_data['land_mask']]
            valid_pct = (100*(~np.isnan(land_vals)).sum()/land_vals.size 
                         if land_vals.size > 0 else 0)
            
            per_day_var = np.nanvar(gridded.reshape(n_days, -1), axis=1)
            med_spatial_var = float(np.nanmedian(per_day_var))
            
            is_real = (n_ok > n_avg) and (med_spatial_var > 1e-4)
            quality_flag = "✅ REAL" if is_real else "⚠️ FALLBACK"
            
            try:
                ds = xr.Dataset(
                    data_vars={
                        var_name: (["time","latitude","longitude"], gridded,
                                   {"units": var_cfg['units'],
                                    "long_name": var_cfg['long_name'],
                                    "_FillValue": np.float32(np.nan)}),
                        f"{var_name}_kriging_variance": (
                            ["time","latitude","longitude"], variance,
                            {"units": f"({var_cfg['units']})^2",
                             "_FillValue": np.float32(np.nan)}),
                    },
                    coords={
                        "time": dates.values,
                        "latitude": (["latitude"], region_data['full_lats'],
                                    {"units":"degrees_north","axis":"Y"}),
                        "longitude": (["longitude"], region_data['full_lons'],
                                     {"units":"degrees_east","axis":"X"}),
                    },
                    attrs={
                        "title": f"USCRN Gridded {var_cfg['long_name']} — {region_name}",
                        "region": region_name,
                        "interpolation_method": method_label,
                        "model": model,
                        "spatial_resolution": "2 degrees",
                        "temporal_resolution": "daily",
                        "temporal_coverage": f"{dates.iloc[0].strftime('%Y-%m-%d')} to {dates.iloc[-1].strftime('%Y-%m-%d')}",
                        "n_stations_used": len(v_cols),
                        "valid_data_fraction": round(valid_pct/100, 4),
                        "n_days_kriged": n_ok,
                        "n_days_averaged": n_avg,
                        "n_days_skipped": n_skip,
                        "median_spatial_variance": round(med_spatial_var, 6),
                        "quality_flag": "real_kriging" if is_real else "mostly_fallback",
                        "creation_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                        "conventions": "CF-1.8",
                        "source": "USCRN station observations from NOAA",
                        "creator": "GNR 640 Mini Project — Geospatial Statistics",
                        "license": "CC-BY 4.0",
                    }
                )
                ds.to_netcdf(output_file, encoding={
                    var_name: {"zlib": True, "complevel": 4},
                    f"{var_name}_kriging_variance": {"zlib": True, "complevel": 4},
                })
                ds.close()
                
                sz = os.path.getsize(output_file)/(1024**2)
                elapsed = (time.time() - t_start)/60
                print(f"     {quality_flag} | {sz:.1f} MB | {elapsed:.1f} min | "
                      f"{n_ok} kriged + {n_avg} avg | spatial_var={med_spatial_var:.4f}")
                
                gen_log.append({
                    "region": region_name, "variable": var_name, "model": model,
                    "method": method_label, "size_mb": round(sz,1),
                    "valid_pct": round(valid_pct,1), "elapsed_min": round(elapsed,2),
                    "n_stations": len(v_cols), "n_kriged": n_ok, "n_averaged": n_avg,
                    "median_spatial_var": round(med_spatial_var,6),
                    "quality": "real_kriging" if is_real else "mostly_fallback",
                    "file": output_file, "skipped": False, "is_fallback": not is_real})
            except Exception as e:
                print(f"     ❌ Save failed: {e}")

# Save logs
gen_df = pd.DataFrame(gen_log)
gen_df.to_csv(os.path.join(STATS_DIR, "section5a_generation_log.csv"), index=False)
if skipped_log:
    pd.DataFrame(skipped_log).to_csv(
        os.path.join(STATS_DIR, "section5a_skipped.csv"), index=False)

elapsed_total = (time.time() - start_total)/60
total_size = sum(g['size_mb'] for g in gen_log)
n_real = sum(1 for g in gen_log if not g.get('is_fallback', False))

print(f"\n{'='*70}")
print(f"SECTION 5a COMPLETE")
print(f"{'='*70}")
print(f"  ⏱️  Time: {elapsed_total:.1f} min")
print(f"  📁 Files: {len(gen_log)} ({total_size:.1f} MB) | Real kriging: {n_real}")

print(f"\n  Per-region:")
for region in ['CONUS', 'Alaska', 'Hawaii']:
    sub = gen_df[gen_df['region'] == region]
    if len(sub) == 0: continue
    real = (~sub['is_fallback'].astype(bool)).sum() if 'is_fallback' in sub.columns else len(sub)
    print(f"     {region:<10} {len(sub)} files ({sub['size_mb'].sum():.1f} MB) — {real} real")

print(f"\n  ✅ Ready for Section 5b")

In [28]:

import os, time, glob
import numpy as np
import pandas as pd
import xarray as xr
from datetime import datetime
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("=" * 70)
print("PRECIPITATION RE-KRIGE (overshoot-protected log-transform)")
print("=" * 70)

# Pre-flight
required = ['VARIABLE_CONFIG','MIN_STATIONS','regions','stations',
            'STATION_KEY','GT_DIR','GRIDDED_DIR','STATS_DIR',
            'krige_day','idw_interp','simple_avg']
missing = [v for v in required if v not in globals()]
if missing: raise RuntimeError(f"Missing: {missing}")

TARGET_VAR = 'Precipitation'
TOP_MODELS_PER_VAR = {'Precipitation': ['gaussian', 'spherical']}
SKIP_COMBINATIONS = {('Alaska', 'SoilMoisture'), ('Hawaii', 'SoilMoisture')}

# ── DELETE old precipitation files ──
old_files = glob.glob(os.path.join(GRIDDED_DIR, "Gridded_Precipitation_*.nc"))
print(f"\n  🗑️  Deleting {len(old_files)} old precipitation files:")
for fp in old_files:
    sz = os.path.getsize(fp) / (1024**2)
    print(f"     Removed: {os.path.basename(fp)} ({sz:.1f} MB)")
    os.remove(fp)

print(f"\n  🔬 Method: log(1+x) → krige → CLIP log to [0, log_max+0.5] → expm1 → clip[0,500]")
print(f"  🛡️ Overshoot protection prevents extrapolation explosion in log-space")

def validate_kriging_output(z, threshold=1e-4):
    valid = z[~np.isnan(z)]
    if len(valid) < 2: return False
    return float(np.ptp(valid)) > threshold

def krige_precipitation_logtransform_safe(v_lons, v_lats, vals, region_data, model):
    """Log-transform kriging with overshoot protection."""
    vals_clean = np.maximum(vals, 0)
    vals_log = np.log1p(vals_clean)
    
    z_log, ss_log = krige_day(v_lons, v_lats, vals_log, region_data, model)
    if np.all(np.isnan(z_log)):
        return z_log, ss_log
    
    # 🛡️ Cap log predictions at observed log_max + 0.5 buffer
    max_observed_log = float(np.nanmax(vals_log))
    log_ceiling = max_observed_log + 0.5
    z_log_clipped = np.clip(z_log, 0, log_ceiling)
    
    z = np.expm1(z_log_clipped)
    z = np.maximum(z, 0)
    return z.astype(np.float32), ss_log.astype(np.float32)

# ── Generation loop ──
start_total = time.time()
gen_log = []

for region_name in ['CONUS', 'Alaska', 'Hawaii']:
    region_data = regions[region_name]
    region_st = stations[stations['Region'] == region_name]
    use_idw = (region_name == 'Hawaii')
    
    print(f"\n{'█'*70}")
    print(f"  📍 {region_name} | {len(region_st)} stations | "
          f"{region_data['land_mask'].sum()} land cells")
    print(f"{'█'*70}")
    
    if (region_name, TARGET_VAR) in SKIP_COMBINATIONS:
        print(f"  ⚠️ SKIP {TARGET_VAR}: in exclusion list"); continue
    
    var_cfg = VARIABLE_CONFIG[TARGET_VAR]
    models_to_run = ['idw'] if use_idw else TOP_MODELS_PER_VAR[TARGET_VAR]
    
    for model in models_to_run:
        t_start = time.time()
        
        if use_idw:
            output_file = os.path.join(GRIDDED_DIR,
                f"Gridded_{TARGET_VAR}_{region_name}_idw_2deg_daily.nc")
            method_label = "IDW"
        else:
            output_file = os.path.join(GRIDDED_DIR,
                f"Gridded_{TARGET_VAR}_{region_name}_{model}_2deg_daily.nc")
            method_label = f"OK ({model}, log+clip)"
        
        print(f"\n  📊 {TARGET_VAR} ({var_cfg['units']}) — {method_label}")
        
        # Load data
        df = pd.read_excel(os.path.join(GT_DIR, var_cfg['file']))
        date_col = df.columns[0]
        st_cols = df.columns[1:]
        df[st_cols] = df[st_cols].apply(pd.to_numeric, errors='coerce')
        df[st_cols] = var_cfg['convert'](df[st_cols])
        
        # Match stations
        v_lons, v_lats, v_cols = [], [], []
        for col in st_cols:
            m = region_st[region_st[STATION_KEY] == col]
            if len(m) > 0:
                v_lons.append(float(m['Lon'].values[0]))
                v_lats.append(float(m['Lat'].values[0]))
                v_cols.append(col)
        v_lons, v_lats = np.array(v_lons), np.array(v_lats)
        if len(v_cols) == 0: print(f"     ⚠️ No stations matched"); continue
        print(f"     Stations matched: {len(v_cols)}")
        
        valid_per_day = df[v_cols].notna().sum(axis=1)
        min_st = MIN_STATIONS[region_name]
        n_usable = ((valid_per_day >= min_st).sum() if not use_idw 
                    else (valid_per_day >= 1).sum())
        print(f"     Usable days: {n_usable}/{len(df)} ({100*n_usable/len(df):.1f}%)")
        
        if n_usable < 30:
            print(f"     ⚠️ Too few usable days"); continue
        
        dates = pd.to_datetime(df[date_col])
        n_days = len(dates)
        n_lat = len(region_data['full_lats'])
        n_lon = len(region_data['full_lons'])
        
        gridded = np.full((n_days, n_lat, n_lon), np.nan, dtype=np.float32)
        variance = np.full((n_days, n_lat, n_lon), np.nan, dtype=np.float32)
        n_ok = n_avg = n_skip = 0
        
        for t in tqdm(range(n_days), desc=f"     Days", leave=False):
            try:
                row = df.iloc[t]
                vals = np.array([float(row[c]) if pd.notna(row[c]) else np.nan
                                 for c in v_cols])
                valid = (~np.isnan(vals)).sum()
                if valid < 1: n_skip += 1; continue
                
                if use_idw:
                    z = idw_interp(v_lons, v_lats, vals, region_data)
                    z = np.maximum(z, 0)
                    ss = np.zeros_like(z)
                    if not np.all(np.isnan(z)): n_ok += 1
                    else: n_skip += 1; continue
                elif valid >= min_st and valid >= 5:
                    z, ss = krige_precipitation_logtransform_safe(
                        v_lons, v_lats, vals, region_data, model)
                    if np.all(np.isnan(z)) or not validate_kriging_output(z):
                        z, ss = simple_avg(vals, region_data)
                        z = np.maximum(z, 0); n_avg += 1
                    else:
                        n_ok += 1
                else:
                    z, ss = simple_avg(vals, region_data)
                    z = np.maximum(z, 0); n_avg += 1
                
                gridded[t] = z; variance[t] = ss
            except Exception:
                n_skip += 1
        
        # Final clip
        gridded = np.clip(gridded, 0, var_cfg['clip_max'])
        
        land_vals = gridded[:, region_data['land_mask']]
        valid_pct = (100*(~np.isnan(land_vals)).sum()/land_vals.size 
                     if land_vals.size > 0 else 0)
        per_day_var = np.nanvar(gridded.reshape(n_days, -1), axis=1)
        med_spatial_var = float(np.nanmedian(per_day_var))
        gridded_mean = float(np.nanmean(gridded))
        gridded_max = float(np.nanmax(gridded))
        raw_mean = float(np.nanmean(df[v_cols].values))
        raw_max = float(np.nanmax(df[v_cols].values))
        bias_ratio = gridded_mean / raw_mean if raw_mean > 0 else 0
        n_clipped = int((gridded >= var_cfg['clip_max']*0.99).sum())
        is_real = (n_ok > n_avg) and (med_spatial_var > 1e-6) and (bias_ratio < 2.0)
        
        ds = xr.Dataset(
            data_vars={
                TARGET_VAR: (["time","latitude","longitude"], gridded,
                           {"units": var_cfg['units'],
                            "long_name": var_cfg['long_name'],
                            "_FillValue": np.float32(np.nan)}),
                f"{TARGET_VAR}_kriging_variance": (
                    ["time","latitude","longitude"], variance,
                    {"units": "log_space" if not use_idw else "(mm/day)^2",
                     "_FillValue": np.float32(np.nan)}),
            },
            coords={
                "time": dates.values,
                "latitude": (["latitude"], region_data['full_lats'],
                            {"units":"degrees_north","axis":"Y"}),
                "longitude": (["longitude"], region_data['full_lons'],
                             {"units":"degrees_east","axis":"X"}),
            },
            attrs={
                "title": f"USCRN Gridded {var_cfg['long_name']} — {region_name}",
                "region": region_name,
                "interpolation_method": method_label,
                "model": model,
                "transform": "log(1+x) → krige → clip log to log_max+0.5 → expm1 → clip[0,500]",
                "raw_station_mean_mmday": round(raw_mean, 4),
                "gridded_mean_mmday": round(gridded_mean, 4),
                "raw_station_max_mmday": round(raw_max, 4),
                "gridded_max_mmday": round(gridded_max, 4),
                "gridded_to_raw_ratio": round(bias_ratio, 4),
                "n_cells_at_clip_max": n_clipped,
                "spatial_resolution": "2 degrees",
                "temporal_resolution": "daily",
                "temporal_coverage": f"{dates.iloc[0].strftime('%Y-%m-%d')} to {dates.iloc[-1].strftime('%Y-%m-%d')}",
                "n_stations_used": len(v_cols),
                "valid_data_fraction": round(valid_pct/100, 4),
                "n_days_kriged": n_ok,
                "n_days_averaged": n_avg,
                "median_spatial_variance": round(med_spatial_var, 6),
                "quality_flag": "real_kriging" if is_real else "needs_review",
                "creation_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "conventions": "CF-1.8",
                "license": "CC-BY 4.0",
            }
        )
        ds.to_netcdf(output_file, encoding={
            TARGET_VAR: {"zlib": True, "complevel": 4},
            f"{TARGET_VAR}_kriging_variance": {"zlib": True, "complevel": 4},
        })
        ds.close()
        
        sz = os.path.getsize(output_file)/(1024**2)
        elapsed = (time.time() - t_start)/60
        quality = "✅ GOOD" if bias_ratio < 1.3 else "⚠️ HIGH BIAS"
        print(f"     {quality} | {sz:.1f} MB | {elapsed:.1f} min")
        print(f"     📊 Raw mean: {raw_mean:.3f} | Gridded mean: {gridded_mean:.3f} | ratio: {bias_ratio:.2f}x")
        print(f"     📊 Raw max:  {raw_max:.1f}  | Gridded max:  {gridded_max:.1f}  | clipped: {n_clipped}")
        
        gen_log.append({
            "region": region_name, "variable": TARGET_VAR, "model": model,
            "method": method_label, "size_mb": round(sz,1),
            "valid_pct": round(valid_pct,1), "elapsed_min": round(elapsed,2),
            "n_stations": len(v_cols), "n_kriged": n_ok, "n_averaged": n_avg,
            "raw_mean": round(raw_mean,4), "gridded_mean": round(gridded_mean,4),
            "bias_ratio": round(bias_ratio,4), "n_clipped": n_clipped,
            "median_spatial_var": round(med_spatial_var,6),
            "quality": "real_kriging" if is_real else "needs_review",
            "file": output_file, "skipped": False, "is_fallback": not is_real})

# Update generation log
gen_log_file = os.path.join(STATS_DIR, "section5a_generation_log.csv")
existing = pd.read_csv(gen_log_file) if os.path.exists(gen_log_file) else pd.DataFrame()
existing = existing[existing['variable'] != TARGET_VAR]
combined = pd.concat([existing, pd.DataFrame(gen_log)], ignore_index=True)
combined.to_csv(gen_log_file, index=False)

elapsed_total = (time.time() - start_total)/60
print(f"\n{'='*70}")
print(f"✅ PRECIPITATION RE-KRIGE COMPLETE")
print(f"{'='*70}")
print(f"  ⏱️  Time: {elapsed_total:.1f} min")
print(f"\n  🔬 Bias verification (target: ratio close to 1.0):")
for entry in gen_log:
    flag = "✅" if entry['bias_ratio'] < 1.3 else "⚠️"
    print(f"     {flag} {entry['region']:<10} {entry['model']:<10} "
          f"ratio={entry['bias_ratio']:.2f}x  clipped={entry['n_clipped']}")
print(f"\n  ✅ Ready for Section 5b")

### SECTION 5b: MODEL SELECTION via ERA5 (RMSE + Correlation)

In [29]:
import os, json
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from scipy import stats as scipy_stats
from scipy.interpolate import griddata

print("=" * 70)
print("SECTION 5b: MODEL SELECTION via ERA5")
print("=" * 70)

def find_data_var(ds):
    skip = {'time','valid_time','latitude','longitude','lat','lon','expver','number'}
    for v in ds.data_vars:
        if v.lower() not in skip: return v
    return list(ds.data_vars)[0]
def find_time_coord(ds): return 'valid_time' if 'valid_time' in ds.coords else 'time'
def find_lat_lon(ds): return ('latitude' if 'latitude' in ds.coords else 'lat',
                              'longitude' if 'longitude' in ds.coords else 'lon')

def regrid(src, src_lats, src_lons, tgt_lats, tgt_lons, mask=None):
    if src_lats[0] > src_lats[-1]:
        src_lats = src_lats[::-1]; src = src[::-1, :]
    sg_lon, sg_lat = np.meshgrid(src_lons, src_lats)
    tg_lon, tg_lat = np.meshgrid(tgt_lons, tgt_lats)
    pts = np.column_stack([sg_lon.ravel(), sg_lat.ravel()])
    vals = src.ravel(); valid = ~np.isnan(vals)
    if valid.sum() < 3: return np.full(tg_lon.shape, np.nan)
    out = griddata(pts[valid], vals[valid], (tg_lon, tg_lat), method='linear')
    if mask is not None: out[~mask] = np.nan
    return out

def compute_metrics(uscrn_file, era5_file, var_name, var_cfg, region_data):
    try:
        ds_u = xr.open_dataset(uscrn_file)
        u_mon = ds_u[var_name].resample(time='MS').mean()
        u_per = pd.to_datetime(u_mon.time.values).to_period('M')
        ds_e = xr.open_dataset(era5_file)
        tc, ev = find_time_coord(ds_e), find_data_var(ds_e)
        lat_c, lon_c = find_lat_lon(ds_e)
        ed = ds_e[ev]
        if 'expver' in ed.dims: ed = ed.isel(expver=0)
        if 'number' in ed.dims: ed = ed.isel(number=0)
        e_per = pd.to_datetime(ds_e[tc].values).to_period('M')
        
        all_u, all_e = [], []
        for p in np.intersect1d(u_per, e_per):
            ui = np.where(u_per == p)[0]; ei = np.where(e_per == p)[0]
            if len(ui)==0 or len(ei)==0: continue
            ug = u_mon.values[ui[0]]; er = ed.values[ei[0]]
            while er.ndim > 2: er = er[0]
            ec = var_cfg['era5_convert'](er.astype(float))
            erg = regrid(ec, ds_e[lat_c].values, ds_e[lon_c].values,
                        region_data['full_lats'], region_data['full_lons'],
                        region_data['land_mask'])
            ul = ug[region_data['land_mask']]; el = erg[region_data['land_mask']]
            mask = ~np.isnan(ul) & ~np.isnan(el)
            all_u.extend(ul[mask]); all_e.extend(el[mask])
        ds_u.close(); ds_e.close()
        
        if len(all_u) < 30: return None
        u, e = np.array(all_u), np.array(all_e)
        rmse = np.sqrt(np.mean((e-u)**2))
        r, r_p = scipy_stats.pearsonr(u, e)
        return {"RMSE": round(rmse,4), "MAE": round(np.mean(np.abs(e-u)),4),
                "Bias": round(np.mean(e-u),4), "Correlation": round(r,4),
                "Correlation_pvalue": float(f"{r_p:.2e}"),
                "N_comparisons": len(u)}
    except Exception as ex:
        return {"error": str(ex)}

gen_df = pd.read_csv(os.path.join(STATS_DIR, "section5a_generation_log.csv"))
print(f"\n  📁 {len(gen_df)} files to evaluate\n")

results = []
for _, row in gen_df.iterrows():
    region, var_name, model = row['region'], row['variable'], row['model']
    region_data = regions[region]
    var_cfg = VARIABLE_CONFIG[var_name]
    era5_file = os.path.join(ERA5_DIR, region,
        f"ERA5_{var_name}_{region}_monthly_2006_2021.nc")
    if not os.path.exists(era5_file):
        print(f"  ⏭ {region:<8} {var_name:<22} [{model:<12}] ERA5 missing"); continue
    print(f"  📊 {region:<8} {var_name:<22} [{model:<12}]...", end=" ", flush=True)
    m = compute_metrics(row['file'], era5_file, var_name, var_cfg, region_data)
    if m and 'RMSE' in m:
        results.append({"Region": region, "Variable": var_name, "Model": model,
                       "Quality": row.get('quality', 'unknown'), **m})
        print(f"RMSE={m['RMSE']:>8.3f}  r={m['Correlation']:>+6.3f}")
    else:
        print(f"failed")

sel_df = pd.DataFrame(results)
sel_df.to_csv(os.path.join(STATS_DIR, "section5b_model_selection.csv"), index=False)

# Pick winners (prefer real_kriging)
print(f"\n{'='*70}\nWINNERS\n{'='*70}\n")
print(f"  {'Region':<10} {'Variable':<22} {'Best Model':<14} {'RMSE':<10} {'r':<8} {'Quality'}")
print(f"  {'─'*10} {'─'*22} {'─'*14} {'─'*10} {'─'*8} {'─'*8}")

WINNERS = {}
SKIP_COMBINATIONS = {('Alaska', 'SoilMoisture')}
for region in ['CONUS', 'Alaska', 'Hawaii']:
    for var_name in VARIABLE_CONFIG:
        if (region, var_name) in SKIP_COMBINATIONS:
            print(f"  {region:<10} {var_name:<22} (excluded)"); continue
        sub = sel_df[(sel_df['Region']==region) & (sel_df['Variable']==var_name)]
        if len(sub) == 0:
            print(f"  {region:<10} {var_name:<22} (no data)"); continue
        real_sub = sub[sub['Quality'] == 'real_kriging']
        chosen = real_sub if len(real_sub) > 0 else sub
        w = chosen.loc[chosen['RMSE'].idxmin()]
        WINNERS[(region, var_name)] = w['Model']
        print(f"  {region:<10} {var_name:<22} {w['Model']:<14} "
              f"{w['RMSE']:<10.3f} {w['Correlation']:<+8.3f} {w['Quality']}")

with open(os.path.join(STATS_DIR, "best_models_final.json"), 'w') as f:
    json.dump({
        'WINNERS': {f"{k[0]}_{k[1]}": v for k, v in WINNERS.items()},
        'SKIPPED': [f"{r}_{v}" for r, v in SKIP_COMBINATIONS],
    }, f, indent=2)

# Plot
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for i, region in enumerate(['CONUS', 'Alaska', 'Hawaii']):
    ax = axes[i]
    sub = sel_df[sel_df['Region'] == region]
    if len(sub) == 0:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(region); continue
    pivot = sub.pivot_table(index='Variable', columns='Model', values='RMSE')
    pivot.plot(kind='bar', ax=ax, edgecolor='black', alpha=0.85)
    ax.set_title(f'{region}: RMSE per Model', fontweight='bold')
    ax.set_ylabel('RMSE (vs ERA5)')
    ax.legend(title='Model', fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')
    ax.tick_params(axis='x', rotation=30)
plt.suptitle('Model Selection via ERA5 RMSE (AIC-justified pairs)',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "section5b_model_selection.svg"),
            format='svg', bbox_inches='tight')
plt.savefig(os.path.join(PLOTS_DIR, "section5b_model_selection.png"),
            dpi=200, bbox_inches='tight')
plt.show()

# AIC vs RMSE concordance analysis
print(f"\n{'='*70}\nAIC vs RMSE CONCORDANCE\n{'='*70}\n")
aic_top1 = {
    'AirTemperature': 'gaussian', 'Precipitation': 'gaussian',
    'RelativeHumidity': 'spherical',  # AIC#1=circular unstable, #2=spherical
    'SoilTemperature': 'gaussian', 'SoilMoisture': 'exponential',
}
print(f"  {'Variable':<22} {'AIC #1':<14} {'CONUS RMSE Winner':<20} {'Match?'}")
print(f"  {'─'*22} {'─'*14} {'─'*20} {'─'*8}")
match = 0; total = 0
for v in VARIABLE_CONFIG:
    if ('CONUS', v) in WINNERS:
        rmse_win = WINNERS[('CONUS', v)]
        aic_w = aic_top1.get(v, '—')
        is_match = (aic_w == rmse_win)
        if is_match: match += 1
        total += 1
        print(f"  {v:<22} {aic_w:<14} {rmse_win:<20} {'✅' if is_match else '⚠️ DIFFER'}")
print(f"\n  Concordance: {match}/{total} = {100*match/total:.0f}%")
print(f"  📌 Disagreement validates the value of independent (ERA5) selection")

print(f"\n  ✅ Section 5b complete")

### SECTION 6: STATISTICAL ANALYSIS 

In [30]:

import os, json
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from scipy import stats as scipy_stats
from scipy.interpolate import griddata

print("=" * 70)
print("SECTION 6: STATISTICAL ANALYSIS")
print("=" * 70)

with open(os.path.join(STATS_DIR, "best_models_final.json")) as f:
    cfg = json.load(f)
WINNERS = {tuple(k.rsplit('_', 1)): v for k, v in cfg['WINNERS'].items()}

def get_winner_file(region, var_name):
    if region == 'Hawaii':
        return os.path.join(GRIDDED_DIR,
            f"Gridded_{var_name}_{region}_idw_2deg_daily.nc")
    model = WINNERS.get((region, var_name))
    if model is None: return None
    return os.path.join(GRIDDED_DIR,
        f"Gridded_{var_name}_{region}_{model}_2deg_daily.nc")

def find_data_var(ds):
    skip = {'time','valid_time','latitude','longitude','lat','lon','expver','number'}
    for v in ds.data_vars:
        if v.lower() not in skip: return v
    return list(ds.data_vars)[0]
def find_time_coord(ds): return 'valid_time' if 'valid_time' in ds.coords else 'time'
def find_lat_lon(ds): return ('latitude' if 'latitude' in ds.coords else 'lat',
                              'longitude' if 'longitude' in ds.coords else 'lon')
def regrid(src, src_lats, src_lons, tgt_lats, tgt_lons, mask=None):
    if src_lats[0] > src_lats[-1]:
        src_lats = src_lats[::-1]; src = src[::-1, :]
    sg_lon, sg_lat = np.meshgrid(src_lons, src_lats)
    tg_lon, tg_lat = np.meshgrid(tgt_lons, tgt_lats)
    pts = np.column_stack([sg_lon.ravel(), sg_lat.ravel()])
    vals = src.ravel(); valid = ~np.isnan(vals)
    if valid.sum() < 3: return np.full(tg_lon.shape, np.nan)
    out = griddata(pts[valid], vals[valid], (tg_lon, tg_lat), method='linear')
    if mask is not None: out[~mask] = np.nan
    return out

results = []
for region in ['CONUS', 'Alaska', 'Hawaii']:
    region_data = regions[region]
    print(f"\n  📍 {region}")
    for var_name, var_cfg in VARIABLE_CONFIG.items():
        uf = get_winner_file(region, var_name)
        ef = os.path.join(ERA5_DIR, region,
            f"ERA5_{var_name}_{region}_monthly_2006_2021.nc")
        if not uf or not os.path.exists(uf) or not os.path.exists(ef):
            print(f"     ⏭ {var_name}: file missing"); continue
        try:
            ds_u = xr.open_dataset(uf)
            u_mon = ds_u[var_name].resample(time='MS').mean()
            u_per = pd.to_datetime(u_mon.time.values).to_period('M')
            ds_e = xr.open_dataset(ef)
            tc, ev = find_time_coord(ds_e), find_data_var(ds_e)
            lat_c, lon_c = find_lat_lon(ds_e)
            ed = ds_e[ev]
            if 'expver' in ed.dims: ed = ed.isel(expver=0)
            if 'number' in ed.dims: ed = ed.isel(number=0)
            e_per = pd.to_datetime(ds_e[tc].values).to_period('M')
            au, ae = [], []
            for p in np.intersect1d(u_per, e_per):
                ui = np.where(u_per == p)[0]; ei = np.where(e_per == p)[0]
                if len(ui)==0 or len(ei)==0: continue
                ug = u_mon.values[ui[0]]; er = ed.values[ei[0]]
                while er.ndim > 2: er = er[0]
                ec = var_cfg['era5_convert'](er.astype(float))
                erg = regrid(ec, ds_e[lat_c].values, ds_e[lon_c].values,
                            region_data['full_lats'], region_data['full_lons'],
                            region_data['land_mask'])
                ul = ug[region_data['land_mask']]; el = erg[region_data['land_mask']]
                mask = ~np.isnan(ul) & ~np.isnan(el)
                au.extend(ul[mask]); ae.extend(el[mask])
            ds_u.close(); ds_e.close()
            if len(au) < 30: print(f"     ⏭ {var_name}: insufficient"); continue
            
            u, e = np.array(au), np.array(ae)
            n = min(10000, len(u))
            np.random.seed(42)
            us = np.random.choice(u, n, replace=False)
            es = np.random.choice(e, n, replace=False)
            ks_s, ks_p = scipy_stats.ks_2samp(us, es)
            
            results.append({
                "Region": region, "Variable": var_name, "Units": var_cfg['units'],
                "USCRN_Mean": round(np.mean(u),4), "ERA5_Mean": round(np.mean(e),4),
                "Mean_Difference": round(np.mean(u)-np.mean(e),4),
                "USCRN_Median": round(np.median(u),4), "ERA5_Median": round(np.median(e),4),
                "USCRN_Variance": round(np.var(u),4), "ERA5_Variance": round(np.var(e),4),
                "USCRN_StdDev": round(np.std(u),4), "ERA5_StdDev": round(np.std(e),4),
                "StdDev_Ratio": round(np.std(u)/np.std(e),4) if np.std(e)>0 else np.nan,
                "KS_Statistic": round(ks_s,4),
                "KS_pvalue": float(f"{ks_p:.2e}"),
                "KS_Reject_H0": bool(ks_p < 0.05),
                "N_samples": len(u),
            })
            print(f"     ✅ {var_name:<22} μ_diff={np.mean(u)-np.mean(e):+7.3f}  "
                  f"σ_ratio={np.std(u)/np.std(e):.3f}  KS={ks_s:.3f}")
        except Exception as ex:
            print(f"     ❌ {var_name}: {ex}")

stats_df = pd.DataFrame(results)
stats_df.to_csv(os.path.join(STATS_DIR, "section6_statistical_analysis.csv"), index=False)

fig, axes = plt.subplots(3, 5, figsize=(24, 12))
for r_idx, region in enumerate(['CONUS', 'Alaska', 'Hawaii']):
    for v_idx, (var_name, var_cfg) in enumerate(VARIABLE_CONFIG.items()):
        ax = axes[r_idx, v_idx]
        row = stats_df[(stats_df['Region']==region) & (stats_df['Variable']==var_name)]
        if len(row) == 0:
            ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                    transform=ax.transAxes, fontsize=14, color='gray')
        else:
            r = row.iloc[0]
            x = np.arange(3)
            u_vals = [r['USCRN_Mean'], r['USCRN_Median'], r['USCRN_StdDev']]
            e_vals = [r['ERA5_Mean'], r['ERA5_Median'], r['ERA5_StdDev']]
            ax.bar(x-0.2, u_vals, 0.4, label='USCRN', color='steelblue', alpha=0.85)
            ax.bar(x+0.2, e_vals, 0.4, label='ERA5', color='coral', alpha=0.85)
            ax.set_xticks(x); ax.set_xticklabels(['Mean','Median','StdDev'], fontsize=9)
            ax.legend(fontsize=8); ax.grid(True, alpha=0.3, axis='y')
            ks = "≠" if r['KS_Reject_H0'] else "≈"
            ax.text(0.98, 0.98, f"KS: {ks}", transform=ax.transAxes,
                    ha='right', va='top', fontweight='bold',
                    bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))
        if r_idx == 0: ax.set_title(var_name, fontweight='bold', fontsize=11)
        if v_idx == 0: ax.set_ylabel(f"{region}", fontweight='bold', fontsize=11)

plt.suptitle('USCRN vs ERA5 Statistical Comparison (KS: ≠ different, ≈ similar)',
             fontsize=14, fontweight='bold', y=1.005)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "section6_statistics.svg"),
            format='svg', bbox_inches='tight')
plt.savefig(os.path.join(PLOTS_DIR, "section6_statistics.png"),
            dpi=200, bbox_inches='tight')
plt.show()

print(f"\n  ✅ Section 6 complete — {len(stats_df)} comparisons")

### SECTION 7: SEASONALITY ANALYSIS

In [31]:
import os, json
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

print("=" * 70)
print("SECTION 7: SEASONALITY ANALYSIS")
print("=" * 70)

with open(os.path.join(STATS_DIR, "best_models_final.json")) as f:
    cfg = json.load(f)
WINNERS = {tuple(k.rsplit('_', 1)): v for k, v in cfg['WINNERS'].items()}

def get_winner_file(region, var_name):
    if region == 'Hawaii':
        return os.path.join(GRIDDED_DIR,
            f"Gridded_{var_name}_{region}_idw_2deg_daily.nc")
    model = WINNERS.get((region, var_name))
    if model is None: return None
    return os.path.join(GRIDDED_DIR,
        f"Gridded_{var_name}_{region}_{model}_2deg_daily.nc")

SEASONS = {'DJF (Winter)': [12,1,2], 'MAM (Spring)': [3,4,5],
           'JJA (Summer)': [6,7,8], 'SON (Fall)': [9,10,11]}

seasonal_stats, monthly_stats = [], []

for region in ['CONUS', 'Alaska', 'Hawaii']:
    region_data = regions[region]
    print(f"\n  📍 {region}")
    for var_name, var_cfg in VARIABLE_CONFIG.items():
        uf = get_winner_file(region, var_name)
        if not uf or not os.path.exists(uf): continue
        try:
            ds = xr.open_dataset(uf); data = ds[var_name]
            for season_name, months in SEASONS.items():
                sd = data.where(data.time.dt.month.isin(months), drop=True)
                land = sd.values[:, region_data['land_mask']]
                land = land[~np.isnan(land)]
                if len(land) == 0: continue
                seasonal_stats.append({
                    "Region": region, "Variable": var_name, "Units": var_cfg['units'],
                    "Season": season_name,
                    "Mean": round(np.mean(land),3), "Median": round(np.median(land),3),
                    "Std": round(np.std(land),3),
                    "P10": round(np.percentile(land,10),3),
                    "P90": round(np.percentile(land,90),3),
                    "N_obs": len(land),
                })
            for month in range(1, 13):
                md = data.where(data.time.dt.month == month, drop=True)
                land = md.values[:, region_data['land_mask']]
                land = land[~np.isnan(land)]
                if len(land) == 0: continue
                monthly_stats.append({
                    "Region": region, "Variable": var_name, "Units": var_cfg['units'],
                    "Month": month, "Mean": round(np.mean(land),3),
                    "Std": round(np.std(land),3),
                    "P10": round(np.percentile(land,10),3),
                    "P90": round(np.percentile(land,90),3),
                })
            ds.close()
        except Exception as ex:
            print(f"     ❌ {var_name}: {ex}")
    print(f"     ✅ Done")

seas_df = pd.DataFrame(seasonal_stats)
mon_df = pd.DataFrame(monthly_stats)
seas_df.to_csv(os.path.join(STATS_DIR, "section7_seasonal.csv"), index=False)
mon_df.to_csv(os.path.join(STATS_DIR, "section7_monthly.csv"), index=False)

# Seasonal plot
fig, axes = plt.subplots(3, 5, figsize=(24, 12))
season_colors = {'DJF (Winter)':'#1f77b4','MAM (Spring)':'#2ca02c',
                 'JJA (Summer)':'#d62728','SON (Fall)':'#ff7f0e'}
for r_idx, region in enumerate(['CONUS', 'Alaska', 'Hawaii']):
    for v_idx, (var_name, var_cfg) in enumerate(VARIABLE_CONFIG.items()):
        ax = axes[r_idx, v_idx]
        sub = seas_df[(seas_df['Region']==region) & (seas_df['Variable']==var_name)]
        if len(sub) == 0:
            ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                    transform=ax.transAxes, color='gray', fontsize=14)
        else:
            seasons = sub['Season'].tolist()
            means = sub['Mean'].tolist()
            stds = sub['Std'].tolist()
            colors = [season_colors[s] for s in seasons]
            ax.bar(range(len(seasons)), means, yerr=stds, color=colors,
                   capsize=5, edgecolor='black', alpha=0.85)
            ax.set_xticks(range(len(seasons)))
            ax.set_xticklabels([s.split()[0] for s in seasons], fontsize=9)
            ax.grid(True, alpha=0.3, axis='y')
        if r_idx == 0: ax.set_title(var_name, fontweight='bold', fontsize=11)
        if v_idx == 0: ax.set_ylabel(region, fontweight='bold', fontsize=11)
plt.suptitle('Seasonal Climatology (mean ± std)', fontsize=15, fontweight='bold', y=1.005)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "section7_seasonal.svg"),
            format='svg', bbox_inches='tight')
plt.savefig(os.path.join(PLOTS_DIR, "section7_seasonal.png"),
            dpi=200, bbox_inches='tight')
plt.show()

# Monthly plot
fig, axes = plt.subplots(3, 5, figsize=(24, 12))
month_labels = ['J','F','M','A','M','J','J','A','S','O','N','D']
region_colors = {'CONUS':'steelblue','Alaska':'darkblue','Hawaii':'darkorange'}
for r_idx, region in enumerate(['CONUS', 'Alaska', 'Hawaii']):
    for v_idx, (var_name, var_cfg) in enumerate(VARIABLE_CONFIG.items()):
        ax = axes[r_idx, v_idx]
        sub = mon_df[(mon_df['Region']==region) & (mon_df['Variable']==var_name)]
        if len(sub) == 0:
            ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                    transform=ax.transAxes, color='gray', fontsize=14)
        else:
            sub = sub.sort_values('Month')
            ax.plot(sub['Month'], sub['Mean'], 'o-',
                    color=region_colors[region], linewidth=2, markersize=6)
            ax.fill_between(sub['Month'], sub['P10'], sub['P90'],
                            color=region_colors[region], alpha=0.2)
            ax.set_xticks(range(1, 13))
            ax.set_xticklabels(month_labels, fontsize=9)
            ax.grid(True, alpha=0.3)
        if r_idx == 0: ax.set_title(var_name, fontweight='bold', fontsize=11)
        if v_idx == 0: ax.set_ylabel(region, fontweight='bold', fontsize=11)
plt.suptitle('Monthly Climatology (mean line, 10–90 percentile band)',
             fontsize=15, fontweight='bold', y=1.005)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "section7_monthly.svg"),
            format='svg', bbox_inches='tight')
plt.savefig(os.path.join(PLOTS_DIR, "section7_monthly.png"),
            dpi=200, bbox_inches='tight')
plt.show()

print(f"\n  ✅ Section 7 complete")

## Section 8 ZENODO PACKAGING

In [32]:

import os, json, zipfile, hashlib, shutil
from datetime import datetime

print("=" * 70)
print("SECTION 8: ZENODO PACKAGING (winners only + evidence)")
print("=" * 70)

PACKAGE_DIR = os.path.join(OUTPUT_DIR, "zenodo_package")
os.makedirs(PACKAGE_DIR, exist_ok=True)

# Load winners
with open(os.path.join(STATS_DIR, "best_models_final.json")) as f:
    cfg = json.load(f)
WINNERS = {tuple(k.rsplit('_', 1)): v for k, v in cfg['WINNERS'].items()}
SKIPPED = cfg.get('SKIPPED', ['Alaska_SoilMoisture'])

# Load AIC justification
aic_just = {}
aic_just_file = os.path.join(STATS_DIR, "aic_justification.json")
if os.path.exists(aic_just_file):
    with open(aic_just_file) as f:
        aic_just = json.load(f)

def get_winner_file(region, var_name):
    if region == 'Hawaii':
        return os.path.join(GRIDDED_DIR,
            f"Gridded_{var_name}_{region}_idw_2deg_daily.nc")
    model = WINNERS.get((region, var_name))
    if model is None: return None
    return os.path.join(GRIDDED_DIR,
        f"Gridded_{var_name}_{region}_{model}_2deg_daily.nc")

# Build deliverable list (winners only)
deliverable_files = []
for region in ['CONUS', 'Alaska', 'Hawaii']:
    for var_name in VARIABLE_CONFIG:
        f = get_winner_file(region, var_name)
        if f and os.path.exists(f):
            deliverable_files.append(f)

# Sanity check
expected_count = (3 * 5) - len(SKIPPED)
if len(deliverable_files) != expected_count:
    print(f"  ⚠️ WARNING: expected {expected_count} winner files, got {len(deliverable_files)}")
else:
    print(f"  ✅ All {expected_count} expected winner files present")

def md5_of(fp):
    h = hashlib.md5()
    with open(fp, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b""): h.update(chunk)
    return h.hexdigest()

# Build manifest
manifest = {
    "title": "USCRN Gridded Daily Climate Data 2006-2021 (CONUS, Alaska, Hawaii)",
    "creator": "25D1388",
    "institution": "Indian Institute of Technology Bombay",
    "course": "GNR 640 — Geospatial Statistics (Mini Project)",
    "creation_date": datetime.now().strftime("%Y-%m-%d"),
    "spatial_resolution": "2 degrees (all regions)",
    "temporal_resolution": "Daily",
    "temporal_coverage": "2006-01-01 to 2021-12-31",
    "variables": list(VARIABLE_CONFIG.keys()),
    "interpolation_methods": {
        "CONUS": "Ordinary Kriging (AIC-screened, ERA5-RMSE selected)",
        "Alaska": "Ordinary Kriging (AIC-screened, ERA5-RMSE selected)",
        "Hawaii": "Inverse Distance Weighting (only 2 stations available)",
    },
    "winning_models": {f"{k[0]}_{k[1]}": v for k, v in WINNERS.items()},
    "model_pairs_evaluated": aic_just.get('model_pairs', {}),
    "aic_justification": aic_just.get('aic_justification', {}),
    "methodology_summary": aic_just.get('methodology_summary',
        "Two-stage selection: AIC top-1 + spherical anchor; final by ERA5 RMSE."),
    "excluded_combinations": {
        "Alaska_SoilMoisture": "Only 1 of 23 stations has valid SM (AK_Kenai_29_ENE)"
    },
    "validation_method": "ERA5 reanalysis (monthly): RMSE + Pearson correlation",
    "license": "CC-BY 4.0",
    "files": [],
}

# Copy winner NetCDFs
print(f"\n  📦 Copying {len(deliverable_files)} winner files...")
for src in deliverable_files:
    fname = os.path.basename(src)
    dst = os.path.join(PACKAGE_DIR, fname)
    if not os.path.exists(dst): shutil.copy2(src, dst)
    manifest['files'].append({
        "name": fname,
        "size_mb": round(os.path.getsize(dst) / (1024**2), 2),
        "md5": md5_of(dst),
    })
    print(f"     ✅ {fname} ({manifest['files'][-1]['size_mb']} MB)")

# Add evidence files
print(f"\n  📋 Adding methodology evidence files...")
evidence_dir = os.path.join(PACKAGE_DIR, "evidence")
os.makedirs(evidence_dir, exist_ok=True)
evidence_files = [
    ("section5b_model_selection.csv", "ERA5 RMSE comparison per candidate model"),
    ("aic_unified_results.csv",       "AIC ranking of all 5 candidate variograms"),
    ("aic_justification.json",        "Model pair selection reasoning"),
    ("section6_statistical_analysis.csv", "USCRN vs ERA5 distribution comparison (KS test)"),
    ("section7_seasonal.csv",         "Seasonal climatology per (region, variable)"),
    ("section7_monthly.csv",          "Monthly climatology per (region, variable)"),
]
for fname, desc in evidence_files:
    src = os.path.join(STATS_DIR, fname)
    if os.path.exists(src):
        dst = os.path.join(evidence_dir, fname)
        shutil.copy2(src, dst)
        print(f"     ✅ evidence/{fname}: {desc}")
    else:
        print(f"     ⚠️ Missing: {fname} (skipped)")

# Save manifest
with open(os.path.join(PACKAGE_DIR, "MANIFEST.json"), 'w') as f:
    json.dump(manifest, f, indent=2)

# README
readme = f"""# USCRN Gridded Daily Climate Data (2006–2021)

## Overview
Daily 2°-resolution gridded products for 5 hydroclimatic variables across CONUS,
Alaska, and Hawaii, generated from USCRN station observations using a
two-stage geostatistical pipeline.

## Methodology
1. **AIC screening** — 5 candidate variogram models (spherical, exponential,
   gaussian, linear, circular) ranked per variable per region
2. **Stable model pair selection** — AIC top-1 + spherical anchor (replaces
   numerically-unstable circular for n=207 CONUS stations)
3. **Daily kriging / IDW** — both candidates evaluated across all 5,844 days
4. **ERA5 RMSE validation** — independent winner selection per (region × variable)
5. **Spatial-variance check** — confirms outputs are genuine spatial fields

## Variables
| Variable | Units | Description |
|----------|-------|-------------|
| AirTemperature | °C | Near-surface air temperature |
| Precipitation | mm | Daily total precipitation |
| RelativeHumidity | % | Near-surface relative humidity |
| SoilTemperature | °C | Soil temperature (5 cm) |
| SoilMoisture | m³/m³ | Soil moisture (10 cm) |

## Files
- **{len(manifest['files'])} NetCDF files** (CF-1.8 compliant, the winning gridded products)
- **MANIFEST.json** — full file inventory + AIC justification + MD5 checksums
- **evidence/** — supporting CSVs documenting model selection and statistical analysis

## License
CC-BY 4.0 — free reuse with attribution.
"""

with open(os.path.join(PACKAGE_DIR, "README.md"), 'w') as f:
    f.write(readme)

# Create zip
zip_path = os.path.join(OUTPUT_DIR, "USCRN_Gridded_2006_2021_FINAL.zip")
print(f"\n  📦 Creating ZIP: {zip_path}")
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for root, _, files in os.walk(PACKAGE_DIR):
        for fname in files:
            fp = os.path.join(root, fname)
            zf.write(fp, os.path.relpath(fp, PACKAGE_DIR))

zip_mb = os.path.getsize(zip_path) / (1024**2)

print(f"\n{'='*70}")
print(f"✅ ZENODO PACKAGE READY")
print(f"{'='*70}")
print(f"  📦 ZIP: {zip_path} ({zip_mb:.1f} MB)")
print(f"  📁 Contents:")
print(f"     • {len(manifest['files'])} NetCDF winner files")
print(f"     • MANIFEST.json (with AIC justification + checksums)")
print(f"     • README.md")
print(f"     • evidence/ folder with {len(evidence_files)} supporting CSVs/JSON")
print(f"\n  🌐 Upload at: https://zenodo.org/uploads/new")
print(f"     • License: CC-BY 4.0")
print(f"     • Title: USCRN Gridded Daily Climate Data 2006–2021")

## SECTION 9: VISUAL INTERPRETATION PLOTS

In [33]:

# Self-contained — rebuilds 'regions' and 'VARIABLE_CONFIG'.
# Generates 5-panel diagnostic figures per (region × variable):
#   Panel 1: USCRN gridded spatial climatology (long-term mean)
#   Panel 2: ERA5 spatial climatology (regridded to 2°)
#   Panel 3: Difference map (USCRN − ERA5)
#   Panel 4: Monthly climatology (averaged across all years per
#            calendar month) with ±1σ interannual variability bands.
#            Precipitation displayed as mm/month (cumulative);
#            other variables in their native unit.
#   Panel 5: Scatter plot with RMSE / MAE / Pearson r / N annotated.
# ============================================================

import os, json, gc
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from scipy import stats as scipy_stats
from scipy.interpolate import griddata

print("=" * 70)
print("SECTION 9: VISUAL INTERPRETATION PLOTS (matched extent)")
print("=" * 70)

# ─────────────────────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────────────────────
BASE_DIR    = "/home/workspace/climatechangeinsights/mini/mini_geospatialstatistics"
OUTPUT_DIR  = os.path.join(BASE_DIR, "output_data_gridded")
GRIDDED_DIR = os.path.join(OUTPUT_DIR, "gridded")
ERA5_DIR    = os.path.join(BASE_DIR, "era5_monthly")
STATS_DIR   = os.path.join(OUTPUT_DIR, "statistics")
PLOTS_DIR   = os.path.join(OUTPUT_DIR, "plots")
PLOTS_VISUAL_DIR = os.path.join(PLOTS_DIR, "visual_interpretation")
os.makedirs(PLOTS_VISUAL_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────
# Auto-detect region grids from gridded NetCDF templates
# ─────────────────────────────────────────────────────────────
def build_regions_from_files():
    regions = {}
    template_files = {
        'CONUS':  'Gridded_AirTemperature_CONUS_spherical_2deg_daily.nc',
        'Alaska': 'Gridded_AirTemperature_Alaska_gaussian_2deg_daily.nc',
        'Hawaii': 'Gridded_AirTemperature_Hawaii_idw_2deg_daily.nc',
    }
    for region, fname in template_files.items():
        fpath = os.path.join(GRIDDED_DIR, fname)
        if not os.path.exists(fpath):
            print(f"  ⚠️  Template not found for {region}: {fname}")
            continue
        with xr.open_dataset(fpath, engine='netcdf4') as ds:
            lats = ds['latitude'].values.copy() if 'latitude' in ds.coords else ds['lat'].values.copy()
            lons = ds['longitude'].values.copy() if 'longitude' in ds.coords else ds['lon'].values.copy()
        regions[region] = {
            'full_lats': lats,
            'full_lons': lons,
            'land_mask': np.ones((len(lats), len(lons)), dtype=bool),
        }
        print(f"  ✅ {region}: {len(lats)} × {len(lons)} grid")
    return regions

VARIABLE_CONFIG = {
    'AirTemperature':   {'era5_convert': lambda x: x - 273.15},
    'Precipitation':    {'era5_convert': lambda x: x * 1000.0},
    'RelativeHumidity': {'era5_convert': lambda x: x},
    'SoilTemperature':  {'era5_convert': lambda x: x - 273.15},
    'SoilMoisture':     {'era5_convert': lambda x: x},
}

print(f"\n  Rebuilding regions from gridded NetCDF templates...")
regions = build_regions_from_files()
print(f"  ✅ Loaded VARIABLE_CONFIG: {list(VARIABLE_CONFIG.keys())}")

with open(os.path.join(STATS_DIR, "best_models_final.json")) as f:
    cfg = json.load(f)
WINNERS = {tuple(k.rsplit('_', 1)): v for k, v in cfg['WINNERS'].items()}
print(f"  ✅ Loaded {len(WINNERS)} winners from best_models_final.json")

# ─────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────
def find_data_var(ds):
    skip = {'time', 'valid_time', 'latitude', 'longitude',
            'lat', 'lon', 'expver', 'number'}
    for v in ds.data_vars:
        if v.lower() not in skip:
            return v
    return list(ds.data_vars)[0]

def find_time_coord(ds):
    return 'valid_time' if 'valid_time' in ds.coords else 'time'

def find_lat_lon(ds):
    return ('latitude' if 'latitude' in ds.coords else 'lat',
            'longitude' if 'longitude' in ds.coords else 'lon')

def regrid(src, src_lats, src_lons, tgt_lats, tgt_lons, mask=None):
    if src_lats[0] > src_lats[-1]:
        src_lats = src_lats[::-1]
        src = src[::-1, :]
    sg_lon, sg_lat = np.meshgrid(src_lons, src_lats)
    tg_lon, tg_lat = np.meshgrid(tgt_lons, tgt_lats)
    pts = np.column_stack([sg_lon.ravel(), sg_lat.ravel()])
    vals = src.ravel()
    valid = ~np.isnan(vals)
    if valid.sum() < 3:
        return np.full(tg_lon.shape, np.nan)
    out = griddata(pts[valid], vals[valid], (tg_lon, tg_lat), method='linear')
    if mask is not None:
        out[~mask] = np.nan
    return out

def get_winner_file(region, var_name):
    if region == 'Hawaii':
        return os.path.join(GRIDDED_DIR,
            f"Gridded_{var_name}_{region}_idw_2deg_daily.nc")
    model = WINNERS.get((region, var_name))
    if model is None:
        return None
    return os.path.join(GRIDDED_DIR,
        f"Gridded_{var_name}_{region}_{model}_2deg_daily.nc")

VAR_PLOT = {
    'AirTemperature':   {'cmap': 'RdYlBu_r', 'label': 'Air Temperature (°C)',
                         'short': 'T_air'},
    'Precipitation':    {'cmap': 'Blues',    'label': 'Precipitation (mm/day)',
                         'short': 'Precip'},
    'RelativeHumidity': {'cmap': 'BrBG',     'label': 'Relative Humidity (%)',
                         'short': 'RH'},
    'SoilTemperature':  {'cmap': 'RdYlBu_r', 'label': 'Soil Temperature (°C)',
                         'short': 'T_soil'},
    'SoilMoisture':     {'cmap': 'YlGnBu',   'label': 'Soil Moisture (m³/m³)',
                         'short': 'SM'},
}

SKIP_COMBINATIONS = {('Alaska', 'SoilMoisture'), ('Hawaii', 'SoilMoisture')}
MONTH_LABELS = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
DAYS_IN_MONTH = np.array([31, 28.25, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31])


# ─────────────────────────────────────────────────────────────
# Build comparison — with COMMON COVERAGE MASK
# ─────────────────────────────────────────────────────────────
def build_comparison(region, var_name):
    region_data = regions[region]
    var_cfg = VARIABLE_CONFIG[var_name]
    
    uf = get_winner_file(region, var_name)
    ef = os.path.join(ERA5_DIR, region,
        f"ERA5_{var_name}_{region}_monthly_2006_2021.nc")
    if not uf or not os.path.exists(uf) or not os.path.exists(ef):
        return None
    
    with xr.open_dataset(uf, engine='netcdf4') as ds_u:
        u_mon = ds_u[var_name].resample(time='MS').mean().load()
    u_per = pd.to_datetime(u_mon.time.values).to_period('M')
    u_mon_values = u_mon.values
    del u_mon
    gc.collect()
    
    with xr.open_dataset(ef, engine='netcdf4') as ds_e:
        tc, ev = find_time_coord(ds_e), find_data_var(ds_e)
        lat_c, lon_c = find_lat_lon(ds_e)
        ed = ds_e[ev]
        if 'expver' in ed.dims: ed = ed.isel(expver=0)
        if 'number' in ed.dims: ed = ed.isel(number=0)
        ed_values = ed.load().values
        e_per = pd.to_datetime(ds_e[tc].values).to_period('M')
        e_lats = ds_e[lat_c].values.copy()
        e_lons = ds_e[lon_c].values.copy()
    del ed
    gc.collect()
    
    common_periods = np.intersect1d(u_per, e_per)
    n_t = len(common_periods)
    n_lat = len(region_data['full_lats'])
    n_lon = len(region_data['full_lons'])
    
    uscrn_3d = np.full((n_t, n_lat, n_lon), np.nan)
    era5_3d  = np.full((n_t, n_lat, n_lon), np.nan)
    months_calendar = []
    years_present = set()
    
    for i, p in enumerate(common_periods):
        ui = np.where(u_per == p)[0]
        ei = np.where(e_per == p)[0]
        if len(ui) == 0 or len(ei) == 0:
            continue
        ug = u_mon_values[ui[0]]
        er = ed_values[ei[0]]
        while er.ndim > 2:
            er = er[0]
        ec = var_cfg['era5_convert'](er.astype(float))
        erg = regrid(ec, e_lats, e_lons,
                     region_data['full_lats'], region_data['full_lons'],
                     region_data['land_mask'])
        uscrn_3d[i] = ug
        era5_3d[i]  = erg
        months_calendar.append(p.month)
        years_present.add(p.year)
    
    months_calendar = np.array(months_calendar)
    year_min, year_max = min(years_present), max(years_present)
    n_years = year_max - year_min + 1
    year_range_str = f"{year_min}-{year_max}"
    
    # Spatial climatology
    uscrn_clim_spatial = np.nanmean(uscrn_3d, axis=0)
    era5_clim_spatial  = np.nanmean(era5_3d,  axis=0)
    
    # ⭐ COMMON COVERAGE MASK — cell valid only if BOTH datasets have data
    common_coverage_mask = (
        ~np.isnan(uscrn_clim_spatial) &
        ~np.isnan(era5_clim_spatial)  &
        region_data['land_mask']
    )
    
    # Apply common mask to BOTH spatial fields
    uscrn_clim_masked = np.where(common_coverage_mask, uscrn_clim_spatial, np.nan)
    era5_clim_masked  = np.where(common_coverage_mask, era5_clim_spatial,  np.nan)
    
    # Monthly climatology — same common mask for both
    uscrn_spatial_mean = np.array([
        np.nanmean(uscrn_3d[i][common_coverage_mask]) for i in range(n_t)
    ])
    era5_spatial_mean = np.array([
        np.nanmean(era5_3d[i][common_coverage_mask])  for i in range(n_t)
    ])
    
    uscrn_monthly_clim = np.zeros(12)
    era5_monthly_clim  = np.zeros(12)
    uscrn_monthly_std  = np.zeros(12)
    era5_monthly_std   = np.zeros(12)
    
    for m in range(1, 13):
        mask = months_calendar == m
        if mask.sum() > 0:
            uscrn_monthly_clim[m-1] = np.nanmean(uscrn_spatial_mean[mask])
            era5_monthly_clim[m-1]  = np.nanmean(era5_spatial_mean[mask])
            uscrn_monthly_std[m-1]  = np.nanstd(uscrn_spatial_mean[mask])
            era5_monthly_std[m-1]   = np.nanstd(era5_spatial_mean[mask])
        else:
            uscrn_monthly_clim[m-1] = np.nan
            era5_monthly_clim[m-1]  = np.nan
    
    # Scatter pairs — common mask applied
    u_flat = uscrn_3d[:, common_coverage_mask].ravel()
    e_flat = era5_3d[:,  common_coverage_mask].ravel()
    valid = ~np.isnan(u_flat) & ~np.isnan(e_flat)
    u_pairs = u_flat[valid]
    e_pairs = e_flat[valid]
    
    del uscrn_3d, era5_3d, u_mon_values, ed_values
    gc.collect()
    
    return {
        'lats': region_data['full_lats'],
        'lons': region_data['full_lons'],
        'common_mask': common_coverage_mask,
        'n_valid_cells': int(common_coverage_mask.sum()),
        'uscrn_clim_spatial': uscrn_clim_masked,
        'era5_clim_spatial':  era5_clim_masked,
        'uscrn_monthly_clim': uscrn_monthly_clim,
        'era5_monthly_clim':  era5_monthly_clim,
        'uscrn_monthly_std':  uscrn_monthly_std,
        'era5_monthly_std':   era5_monthly_std,
        'uscrn_pairs': u_pairs,
        'era5_pairs':  e_pairs,
        'year_range_str': year_range_str,
        'n_years': n_years,
    }


# ─────────────────────────────────────────────────────────────
# Plot generator (uses common mask everywhere)
# ─────────────────────────────────────────────────────────────
def plot_visual_interp(region, var_name, data):
    plot_cfg = VAR_PLOT[var_name]
    year_range_str = data['year_range_str']
    n_years = data['n_years']
    n_cells = data['n_valid_cells']
    
    fig = plt.figure(figsize=(18, 12))
    gs = fig.add_gridspec(2, 3, height_ratios=[1.2, 1],
                          width_ratios=[1, 1, 1.1],
                          hspace=0.30, wspace=0.30)
    
    # Common color scale derived from MASKED data (ignore NaNs from clipping)
    combined = np.concatenate([data['uscrn_clim_spatial'].ravel(),
                                data['era5_clim_spatial'].ravel()])
    combined = combined[~np.isnan(combined)]
    if len(combined) > 0:
        vmin = np.nanpercentile(combined, 2)
        vmax = np.nanpercentile(combined, 98)
    else:
        vmin, vmax = 0, 1
    
    # Panel 1: USCRN spatial (already masked)
    ax1 = fig.add_subplot(gs[0, 0])
    im1 = ax1.pcolormesh(data['lons'], data['lats'],
                         data['uscrn_clim_spatial'],
                         cmap=plot_cfg['cmap'], vmin=vmin, vmax=vmax,
                         shading='auto')
    ax1.set_title(f'USCRN Gridded\n{n_years}-year climatology ({year_range_str})\n'
                  f'{n_cells} valid cells',
                  fontweight='bold', fontsize=10)
    ax1.set_xlabel('Longitude'); ax1.set_ylabel('Latitude')
    plt.colorbar(im1, ax=ax1, label=plot_cfg['label'], fraction=0.046, pad=0.04)
    
    # Panel 2: ERA5 spatial (now masked to USCRN extent)
    ax2 = fig.add_subplot(gs[0, 1])
    im2 = ax2.pcolormesh(data['lons'], data['lats'],
                         data['era5_clim_spatial'],
                         cmap=plot_cfg['cmap'], vmin=vmin, vmax=vmax,
                         shading='auto')
    ax2.set_title(f'ERA5 (regridded, clipped to USCRN extent)\n'
                  f'{n_years}-year climatology ({year_range_str})\n'
                  f'{n_cells} valid cells',
                  fontweight='bold', fontsize=10)
    ax2.set_xlabel('Longitude'); ax2.set_ylabel('Latitude')
    plt.colorbar(im2, ax=ax2, label=plot_cfg['label'], fraction=0.046, pad=0.04)
    
    # Panel 3: Difference
    ax3 = fig.add_subplot(gs[0, 2])
    diff = data['uscrn_clim_spatial'] - data['era5_clim_spatial']
    diff_max = max(abs(np.nanpercentile(diff, 2)), abs(np.nanpercentile(diff, 98)))
    if diff_max == 0 or np.isnan(diff_max):
        diff_max = 1
    im3 = ax3.pcolormesh(data['lons'], data['lats'], diff,
                         cmap='RdBu_r', vmin=-diff_max, vmax=diff_max,
                         shading='auto')
    ax3.set_title(f'Difference (USCRN − ERA5)\n{n_years}-year climatology',
                  fontweight='bold', fontsize=11)
    ax3.set_xlabel('Longitude'); ax3.set_ylabel('Latitude')
    plt.colorbar(im3, ax=ax3, label=f'Δ {plot_cfg["label"]}',
                 fraction=0.046, pad=0.04)
    
    # Panel 4: Monthly climatology
    if var_name == 'Precipitation':
        uscrn_clim_disp = data['uscrn_monthly_clim'] * DAYS_IN_MONTH
        era5_clim_disp  = data['era5_monthly_clim']  * DAYS_IN_MONTH
        uscrn_std_disp  = data['uscrn_monthly_std']  * DAYS_IN_MONTH
        era5_std_disp   = data['era5_monthly_std']   * DAYS_IN_MONTH
        monthly_ylabel  = 'Precipitation (mm/month, cumulative)'
    else:
        uscrn_clim_disp = data['uscrn_monthly_clim']
        era5_clim_disp  = data['era5_monthly_clim']
        uscrn_std_disp  = data['uscrn_monthly_std']
        era5_std_disp   = data['era5_monthly_std']
        monthly_ylabel  = plot_cfg['label']
    
    ax4 = fig.add_subplot(gs[1, :2])
    months = np.arange(1, 13)
    ax4.plot(months, uscrn_clim_disp, 'o-', color='steelblue',
             markersize=8, linewidth=2.2, label='USCRN gridded', zorder=3)
    ax4.fill_between(months,
                     uscrn_clim_disp - uscrn_std_disp,
                     uscrn_clim_disp + uscrn_std_disp,
                     color='steelblue', alpha=0.18, zorder=1,
                     label='USCRN ±1σ (interannual)')
    ax4.plot(months, era5_clim_disp, 's-', color='coral',
             markersize=8, linewidth=2.2, label='ERA5 (regridded)', zorder=3)
    ax4.fill_between(months,
                     era5_clim_disp - era5_std_disp,
                     era5_clim_disp + era5_std_disp,
                     color='coral', alpha=0.18, zorder=1,
                     label='ERA5 ±1σ (interannual)')
    ax4.set_xlabel('Month', fontsize=11)
    ax4.set_ylabel(monthly_ylabel, fontsize=11)
    ax4.set_xticks(months)
    ax4.set_xticklabels(MONTH_LABELS)
    ax4.set_title(f'Monthly Climatology (averaged across {n_years} years, {year_range_str}, '
                  f'spatial mean over {n_cells} common cells)',
                  fontweight='bold', fontsize=10)
    ax4.legend(loc='best', fontsize=9, framealpha=0.9, ncol=2)
    ax4.grid(True, alpha=0.3)
    
    # Panel 5: Scatter
    ax5 = fig.add_subplot(gs[1, 2])
    u_p, e_p = data['uscrn_pairs'], data['era5_pairs']
    n_max_plot = 5000
    if len(u_p) > n_max_plot:
        np.random.seed(42)
        idx = np.random.choice(len(u_p), n_max_plot, replace=False)
        u_plot, e_plot = u_p[idx], e_p[idx]
    else:
        u_plot, e_plot = u_p, e_p
    ax5.scatter(e_plot, u_plot, s=4, alpha=0.25, color='darkblue', edgecolor='none')
    lo = min(np.nanmin(u_p), np.nanmin(e_p))
    hi = max(np.nanmax(u_p), np.nanmax(e_p))
    ax5.plot([lo, hi], [lo, hi], 'r--', linewidth=1.5, label='1:1 line')
    rmse = np.sqrt(np.mean((u_p - e_p) ** 2))
    mae  = np.mean(np.abs(u_p - e_p))
    if np.std(u_p) > 0 and np.std(e_p) > 0:
        r, _ = scipy_stats.pearsonr(u_p, e_p)
    else:
        r = np.nan
    txt = (f"RMSE = {rmse:.3f}\nMAE  = {mae:.3f}\n"
           f"r     = {r:+.3f}\nN     = {len(u_p):,}")
    ax5.text(0.04, 0.96, txt, transform=ax5.transAxes,
             fontsize=10, verticalalignment='top', family='monospace',
             bbox=dict(boxstyle='round', facecolor='lightyellow',
                       edgecolor='black', alpha=0.9))
    ax5.set_xlabel(f'ERA5 {plot_cfg["short"]}')
    ax5.set_ylabel(f'USCRN {plot_cfg["short"]}')
    ax5.set_title(f'Scatter: USCRN vs ERA5\n(common cells, {year_range_str})',
                  fontweight='bold', fontsize=11)
    ax5.legend(loc='lower right', fontsize=9)
    ax5.grid(True, alpha=0.3)
    ax5.set_aspect('equal', adjustable='datalim')
    
    method = "IDW" if region == 'Hawaii' else f"OK ({WINNERS.get((region, var_name), 'N/A')})"
    fig.suptitle(f'{region}: {var_name} — {method}',
                 fontsize=15, fontweight='bold', y=0.995)
    
    out_svg = os.path.join(PLOTS_VISUAL_DIR, f"visual_{region}_{var_name}.svg")
    out_png = os.path.join(PLOTS_VISUAL_DIR, f"visual_{region}_{var_name}.png")
    plt.savefig(out_svg, format='svg', bbox_inches='tight')
    plt.savefig(out_png, dpi=180, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    gc.collect()
    
    return rmse, mae, r, n_years, year_range_str, n_cells


# ─────────────────────────────────────────────────────────────
# Run for all 13 deliverables
# ─────────────────────────────────────────────────────────────
print(f"\n  📁 Output directory: {PLOTS_VISUAL_DIR}\n")

summary = []
for region in ['CONUS', 'Alaska', 'Hawaii']:
    for var_name in VARIABLE_CONFIG:
        if (region, var_name) in SKIP_COMBINATIONS:
            print(f"  ⏭  {region:<8} {var_name:<22} (excluded)")
            continue
        print(f"  📊 {region:<8} {var_name:<22}...", end=" ", flush=True)
        try:
            data = build_comparison(region, var_name)
            if data is None:
                print("file missing")
                continue
            rmse, mae, r, ny, yr, nc = plot_visual_interp(region, var_name, data)
            method_used = "IDW" if region == 'Hawaii' else WINNERS.get((region, var_name), 'N/A')
            summary.append({
                'Region': region, 'Variable': var_name,
                'Method': method_used,
                'Years': ny, 'YearRange': yr,
                'CommonCells': nc,
                'RMSE': round(rmse, 4), 'MAE': round(mae, 4),
                'r': round(r, 4) if not np.isnan(r) else None,
            })
            print(f"✅ {ny}yr ({yr})  cells={nc}  RMSE={rmse:.3f}  MAE={mae:.3f}  r={r:+.3f}")
            del data
            gc.collect()
        except Exception as ex:
            print(f"❌ {ex}")

summary_df = pd.DataFrame(summary)
summary_csv = os.path.join(STATS_DIR, "section9_visual_summary.csv")
summary_df.to_csv(summary_csv, index=False)

print(f"\n{'='*70}")
print(f"✅ VISUAL INTERPRETATION COMPLETE")
print(f"{'='*70}")
print(f"  📊 {len(summary)} figures generated")
print(f"  📁 Plots: {PLOTS_VISUAL_DIR}")
print(f"  📁 Summary CSV: {summary_csv}")
print(f"\n  Summary table:")
print(summary_df.to_string(index=False))